## Variable preparation and benchmark specifications

This notebook documents the preparation of the explanatory variables used in the destination choice model and the sequence of model specifications used as benchmarks for the final proposed specification. The purpose is not to report all numerical results directly in the notebook text, but to make clear how the inputs are prepared, how the estimation samples are constructed, how the sampled choice-set size is selected, and how the different benchmark specifications relate to the final model.

The code cells already contain detailed implementation comments. This markdown section therefore gives the conceptual explanation of the workflow and clarifies the role of each modelling block.

---

### 1. Variable preparation

The first part of the notebook prepares the destination-level variables used as explanatory attributes in the model. The input is the traffic-zone layer containing the spatial units and the destination-side variables. The workflow keeps the relevant metadata columns, selects the variables used in the model, applies transformations where needed, and exports diagnostic tables for checking multicollinearity and correlation.

Most variables are count or density variables. These variables are transformed using `log1p`, which applies the transformation:

\[
\log(1+x)
\]

This is useful because many destination variables are highly skewed and may contain many zero values. The `log1p` transformation preserves zeros while reducing the influence of extreme values. Variables that are already constructed as indices, such as land-use mix, are kept on their original scale when appropriate.

The retained destination-side variables include gastronomy, population, cultural opportunities, sport facilities, outdoor recreation, urban amenities, other leisure opportunities, and super-infrastructure. Together, these variables represent the destination attractivity dimensions used in the final model.

The `VariableAnalysis/` output contains the prepared variable layers and diagnostics. The main prepared layer is the transformed traffic-zone dataset used later in the model estimation. The folder also contains the selected raw variables, the log-transformed variables, the VIF diagnostics, the full correlation matrix, and the sorted pairwise correlation table. These diagnostics are used to check whether the selected variables are suitable for joint estimation and whether strong collinearity is present between candidate explanatory variables.

The version retained for the main workflow is the normal traffic-zone universe used in the thesis model. The variable preparation step is therefore part of the main modelling pipeline and not a separate sensitivity exercise.

---

### 2. Train and out-of-sample split

After the destination variables are prepared, the trip observations are split into training and out-of-sample datasets. The input trip tables contain the observed origin, observed destination, survey year, person identifier, and expansion weight.

Two splitting logics are implemented in the notebook.

The first is an origin-based split, where entire origin zones are assigned either to the training sample or to the out-of-sample sample. This creates a stricter spatial generalization setting, because the model is evaluated on origins that were not used during estimation.

The second is a person-based split, where complete persons are assigned to either training or out-of-sample. This is the preferred split for the final proposed specification because it avoids leakage between training and evaluation when the same person contributes multiple observations. The person-based split also balances the out-of-sample share across survey waves when both 2015 and 2021 are present.

The final proposed specification uses the person-based split. This keeps the evaluation design consistent with the individual-level nature of the trip data while still preserving a meaningful external out-of-sample test.

---

### 3. Choice-set sampling and K selection

The model is estimated as a sampled-choice-set multinomial logit model. For each observed trip, the chosen destination is combined with a fixed number of sampled non-chosen alternatives. The number of sampled alternatives is denoted by `K`.

The notebook includes a dedicated K-selection step. This step uses only the external training data. Within that training data, an additional internal weighted validation split is created. Several candidate values of `K` are then compared using the same model structure.

This step is only used to select the sampled choice-set size. It is not the final model estimation. Once `K` has been chosen, the final proposed model is re-estimated on the full external training sample and evaluated once on the external out-of-sample sample.

This distinction is important because the K-selection step is a tuning step, while the final proposed specification is the actual model used for interpretation and thesis results.

---

### 4. Final proposed specification

The main model in the notebook is the final proposed specification. Earlier labels such as “final baseline” should be understood only as working names from the modelling process. The relevant final model for the thesis is the final proposed specification.

The final proposed model is estimated separately for the four behavioural segments:

- young-short leisure trips;
- old-short leisure trips;
- young-long leisure trips;
- old-long leisure trips.

This segmentation is central to the thesis because the research question concerns differences in leisure destination choice between younger and older people and between short- and long-distance leisure travel. Estimating separate models allows the destination attributes to have different effects across these groups.

The final proposed specification combines three main components.

First, it uses the transformed and standardized destination attractivity variables. Standardization means that coefficients are estimated for one-standard-deviation changes in the transformed variables, which makes the coefficients more comparable across variables.

Second, it includes the EMU accessibility measure derived from the mode-specific utility structure. This term captures the accessibility of each destination from each origin, based on the underlying travel impedance and multimodal utility information.

Third, it uses the person-based train/OOS split and the selected value of `K`, so that the final estimation and evaluation are consistent with the earlier tuning and data-splitting decisions.

The final proposed specification therefore represents the main modelling structure used to interpret destination attractivity in the thesis.

---

### 5. Benchmark specifications

The notebook also includes additional benchmark specifications. These are not robustness checks in this section. They are benchmark models used to understand how the final proposed specification compares to simpler or differently structured alternatives.

The benchmark models are useful because they show what is gained by the final proposed modelling structure. In particular, they help separate the contribution of accessibility, segmentation, and destination attractivity variables.

The accessibility-only baseline keeps the modelling focus on accessibility-related information. This benchmark helps assess how much explanatory power is already captured by accessibility alone, before adding the full destination attractivity structure.

The SIMBA baseline is included to align the model workflow with the later SIMBA integration logic. It is useful because the final model outputs must eventually be translated into destination-level attractivity and utility components that can be used in a simulation environment.

The non-segmented baseline estimates the model without separating the observations into the four age-distance segments. This benchmark is important because it provides a direct comparison against the segmented specification. If the segmented model reveals different coefficient patterns across young/old and short/long groups, the non-segmented baseline helps show why the segmented structure is substantively meaningful.

These benchmark models therefore serve as structured comparisons. They are not the final thesis specification, but they provide context for evaluating the final proposed model.

---

### 6. Hessian-based coefficient inference

The final proposed specification also includes a post-estimation inference step. After the model coefficients are estimated, an approximate Hessian-based variance-covariance matrix is computed for each segment.

This allows the workflow to derive approximate standard errors, z-values, p-values, and coefficient-difference tests. Two types of coefficient comparisons are considered.

The first type compares the same coefficient across different segments. This helps identify whether, for example, the effect of a sport-related destination variable differs between young-short and old-short trips.

The second type compares different coefficients within the same segment. This helps identify whether certain destination attributes have significantly stronger or weaker estimated effects than others within a given behavioural group.

These tests are used as interpretive support. They do not change the estimated model itself; they provide additional evidence for discussing differences between variables and between segments.

---

### 7. Odds-based interpretation

The last interpretive block translates selected standardized coefficients into odds-based quantities. Since the model is a multinomial logit model, a coefficient can be interpreted through its exponential:

\[
\exp(\beta)
\]

This gives the multiplicative change in the odds of choosing a destination associated with a one-unit increase in the corresponding standardized explanatory variable, holding the other variables constant.

Because the explanatory variables are standardized after transformation, a one-unit increase corresponds to a one-standard-deviation increase in the transformed variable. For variables transformed with `log1p`, this means that the interpretation is not a simple one-unit increase in the raw count. The notebook therefore includes an additional step that links a one-standard-deviation increase in transformed space back to approximate changes in the original raw units at different points of the distribution.

The odds-based exploration is included to make selected coefficients easier to interpret substantively. Instead of discussing only abstract standardized coefficients, the notebook reports how the implied odds multiplier changes across segments for selected variables. This is especially useful for comparing whether a given destination attribute has a stronger or weaker behavioural effect for different age-distance groups.

Overall, the odds-based block is an interpretation tool. It complements the coefficient tables and segment comparisons by translating selected effects into more intuitive multiplicative odds changes.

## Preparation: Variable selection, log(1 + x) transformation, correlation analysis

In [ ]:
# ============================================================
# Variable selection + log1p transform + VIF + correlation export
# + PLOTS (Python display only, no saving)
# PLUS: _woLIE version (drops Liechtenstein npvm_id)
# (NO Liechtenstein population correction)
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# ------------------------------------------------------------
# Liechtenstein NPVM IDs to EXCLUDE
# ------------------------------------------------------------
LIE_NPVM = {
    700101001, 700201001, 700301001, 700401001, 700501001, 700601001, 700701001,
    700801001, 700901001, 701001001, 701101001, 710101001, 730101001
}

# ------------------------------------------------------------
# Paths / folders
# ------------------------------------------------------------
TZ_PATH  = "TZ/TZ.gpkg"
TZ_LAYER = "TZ_fixed"

OUT_DIR = "VariableAnalysis"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_GPKG  = os.path.join(OUT_DIR, "TZ_first_sel_log1p.gpkg")
OUT_LAYER = "TZ_first_sel_log1p"

OUT_GPKG_WO  = os.path.join(OUT_DIR, "TZ_first_sel_log1p_woLIE.gpkg")
OUT_LAYER_WO = "TZ_first_sel_log1p_woLIE"

SEL_GPKG  = os.path.join(OUT_DIR, "TZ_sel.gpkg")
SEL_LAYER = "TZ_sel"

SEL_GPKG_WO  = os.path.join(OUT_DIR, "TZ_sel_woLIE.gpkg")
SEL_LAYER_WO = "TZ_sel_woLIE"

# ------------------------------------------------------------
# Columns to keep
# ------------------------------------------------------------
META_COLS = [
    "zone_id","npvm_id","ID_Gem","stg_type","zone_in_gem",
    "Area_m2","Area_km2","ID_alt","N_Gem","N_stg_type","ID_KT","N_KT",
    "ID_SL3","N_SL3","ID_Agglo","N_Agglo","ID_AMR","N_AMR",
]


SEL_VARS = [
    "F1_gastr_count",

    "F2_pop_total",

    "F8_outdoor_lake_raw_dens",

    "F3_outdoor_hard_count",
    "F3_outdoor_soft_count",
    "F3_outdoor_LUmix",


    "F4_cult_count",

    "F5_sport_count",
    "F5_sport_out_length",

    "F6_others_count",

    "F7_urban_sum",

    "F8_POI_urban_dens",

    "F10_superinfra",

    # second wave variables (not in first selection, but we can check them in the same pipeline)

    #"F8_gastr_count_dens",
    #"F8_cult_count_dens",
    #"F8_sport_count_dens",
    #"F8_others_count_dens",
    
]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def safe_log1p(series: pd.Series, colname: str) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").copy()
    n_neg = int((s < 0).sum(skipna=True))
    if n_neg > 0:
        print(f"[WARN] {colname}: found {n_neg} negative values; clipping to 0 before log1p.")
        s = s.clip(lower=0)
    return np.log1p(s)

def compute_vif(df_num: pd.DataFrame) -> pd.DataFrame:
    X = df_num.replace([np.inf, -np.inf], np.nan).dropna(axis=0, how="any").copy()
    variances = X.var(axis=0)
    keep_cols = variances[variances > 1e-12].index.tolist()
    dropped = sorted(set(X.columns) - set(keep_cols))
    if dropped:
        print(f"[INFO] Dropping {len(dropped)} constant/near-constant cols for VIF: {dropped}")
    X = X[keep_cols]

    X_const = sm.add_constant(X, has_constant="add")

    vif_rows = []
    for i, col in enumerate(X_const.columns):
        if col == "const":
            continue
        vif_val = variance_inflation_factor(X_const.values, i)
        vif_rows.append({"variable": col, "vif": float(vif_val)})

    out = pd.DataFrame(vif_rows).sort_values("vif", ascending=False).reset_index(drop=True)
    out["n_rows_used"] = len(X)
    out["n_vars_used"] = len(X.columns)
    return out

def corr_pairs(df_num: pd.DataFrame):
    X = df_num.replace([np.inf, -np.inf], np.nan).dropna(axis=0, how="any")
    corr = X.corr(method="pearson")
    pairs = []
    cols = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            c = corr.iloc[i, j]
            pairs.append({"var1": cols[i], "var2": cols[j], "corr": float(c), "abs_corr": float(abs(c))})
    pairs_df = pd.DataFrame(pairs).sort_values(["abs_corr", "corr"], ascending=[False, False]).reset_index(drop=True)
    pairs_df["n_rows_used"] = len(X)
    return pairs_df, corr

def plot_factor_maps(gdf: gpd.GeoDataFrame, cols: list[str], title_prefix: str = "", figsize=(10, 8)):
    for col in cols:
        fig, ax = plt.subplots(1, 1, figsize=figsize)
        gdf.plot(
            column=col,
            ax=ax,
            legend=True,
            linewidth=0.0,
            edgecolor="none",
            missing_kwds={"color": "lightgrey", "label": "Missing"},
        )
        ax.set_axis_off()
        ax.set_title(f"{title_prefix}{col}", fontsize=13)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

def run_pipeline(gdf_in: gpd.GeoDataFrame, tag: str):
    """
    tag: "" for normal, "_woLIE" for filtered version
    """
    if tag == "_woLIE":
        gdf_work = gdf_in[~gdf_in["npvm_id"].isin(LIE_NPVM)].copy()
    else:
        gdf_work = gdf_in.copy()

    print(f"[{tag or 'BASE'}] rows={len(gdf_work):,}")

    # Keep subset
    gdf_sel = gdf_work[META_COLS + SEL_VARS + ["geometry"]].copy()

    # Write raw selection
    if tag == "_woLIE":
        gdf_sel.to_file(SEL_GPKG_WO, layer=SEL_LAYER_WO, driver="GPKG")
        print(f"[OK] Wrote raw selection: {SEL_GPKG_WO} (layer={SEL_LAYER_WO})")
    else:
        gdf_sel.to_file(SEL_GPKG, layer=SEL_LAYER, driver="GPKG")
        print(f"[OK] Wrote raw selection: {SEL_GPKG} (layer={SEL_LAYER})")

    # log1p
    log_cols = []
    for col in SEL_VARS:
        new_col = f"{col}_log1p"
        gdf_sel[new_col] = safe_log1p(gdf_sel[col], col)
        log_cols.append(new_col)

    # Save raw+log1p
    if tag == "_woLIE":
        gdf_sel.to_file(OUT_GPKG_WO, layer=OUT_LAYER_WO, driver="GPKG")
        print(f"[OK] Wrote raw+log1p: {OUT_GPKG_WO} (layer={OUT_LAYER_WO})")
    else:
        gdf_sel.to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG")
        print(f"[OK] Wrote raw+log1p: {OUT_GPKG} (layer={OUT_LAYER})")

    # VIF + corr
    X_log = gdf_sel[log_cols].copy()
    vif_df = compute_vif(X_log)
    pairs_df, corr_mat = corr_pairs(X_log)

    suf = tag if tag else ""
    vif_df.to_csv(os.path.join(OUT_DIR, f"vif_log1p{suf}.csv"), index=False)
    corr_mat.to_csv(os.path.join(OUT_DIR, f"corr_log1p_matrix{suf}.csv"), index=True)
    pairs_df.to_csv(os.path.join(OUT_DIR, f"corr_log1p_pairs{suf}.csv"), index=False)

    print(f"[OK] Wrote VIF/corr CSVs for tag='{tag or 'BASE'}'")

    # Plots
    plot_factor_maps(gdf_sel, log_cols, title_prefix=f"{tag or 'BASE'}: ", figsize=(10, 8))

    return gdf_sel

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
gdf = gpd.read_file(TZ_PATH, layer=TZ_LAYER)

missing_meta = [c for c in META_COLS if c not in gdf.columns]
missing_vars = [c for c in SEL_VARS if c not in gdf.columns]
if missing_meta or missing_vars:
    raise ValueError(
        "Missing columns in input layer.\n"
        f"Missing META_COLS: {missing_meta}\n"
        f"Missing SEL_VARS: {missing_vars}\n"
        f"Available columns sample: {list(gdf.columns)[:50]}"
    )

# ------------------------------------------------------------
# Run BASE + woLIE
# ------------------------------------------------------------
print("=== BASE version ===")
_ = run_pipeline(gdf, tag="")

print("\n=== woLIE version ===")
_ = run_pipeline(gdf, tag="_woLIE")

print("\n[DONE]")


## Train -- OOS out of sample split

In [ ]:
# ============================================================
# Split trip tables into byOrigin / byPerson
#
# INPUT ASSUMPTION:
#   - Trips/destinations_*.csv have already been regenerated
#     with the CORRECT trip extraction:
#       * YS / OS: short leisure trips = wzweck3 == 8 AND wzweck2 == 1
#       * YL / OL: long leisure trips from Tagesreisen as before
#
# OUTPUT:
#   - byOrigin      : global leave-origins-out
#   - byPerson      : wave-balanced person holdout (50/50 WP across 2015/2021 when both waves exist)
#   - *_woLIE       : same splits after dropping trips with orig OR dest in Liechtenstein NPVM IDs
#
# NOTE:
#   - Existing output CSVs are overwritten.
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

# -----------------------------
# Liechtenstein NPVM IDs to EXCLUDE
# -----------------------------
LIE_NPVM = {
    700101001, 700201001, 700301001, 700401001, 700501001, 700601001, 700701001,
    700801001, 700901001, 701001001, 701101001, 710101001, 730101001
}

# -----------------------------
# CONFIG
# -----------------------------
INPUT_GLOB = "Trips/destinations_*.csv"

OUT_DIR_ORIGIN    = "Trips/byOrigin"
OUT_DIR_PERSON    = "Trips/byPerson"
OUT_DIR_ORIGIN_WO = "Trips/byOrigin_woLIE"
OUT_DIR_PERSON_WO = "Trips/byPerson_woLIE"

TARGET_OOS_SHARE = 0.20
SEED_ORIGIN = 42
SEED_PERSON = 43

WEIGHT_COL_CANDIDATES = ["WP"]
ORIGIN_COL_CANDIDATES = ["orig_zone"]
DEST_COL_CANDIDATES   = ["dest_zone"]
PERSON_COL_CANDIDATES = ["HHNR"]
SURVEY_COL_CANDIDATES = ["survey"]

# -----------------------------
# HELPERS
# -----------------------------
def pick_col(df: pd.DataFrame, candidates: list[str], label: str) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(
        f"Could not find a '{label}' column. "
        f"Tried: {candidates}. Available: {list(df.columns)}"
    )

def get_weight_col(df: pd.DataFrame) -> str:
    wcol = pick_col(df, WEIGHT_COL_CANDIDATES, "weight (WP)")
    df[wcol] = pd.to_numeric(df[wcol], errors="coerce")
    if df[wcol].isna().all():
        raise ValueError(f"Weight column '{wcol}' is all-NaN after numeric conversion.")
    return wcol

def segment_from_filename(path: str) -> str:
    base = os.path.basename(path)
    stem = base.replace(".csv", "")
    # expected: destinations_YS.csv -> YS
    parts = stem.split("_")
    return parts[1] if len(parts) >= 2 else stem

def write_split(out_dir: str, seg: str, train_df: pd.DataFrame, oos_df: pd.DataFrame):
    os.makedirs(out_dir, exist_ok=True)

    train_path = os.path.join(out_dir, f"destinations_{seg}_train.csv")
    oos_path   = os.path.join(out_dir, f"destinations_{seg}_oos.csv")

    train_df.to_csv(train_path, index=False)
    oos_df.to_csv(oos_path, index=False)

    print(f"  -> wrote {train_path} ({len(train_df):,} rows)")
    print(f"  -> wrote {oos_path} ({len(oos_df):,} rows)")

def weighted_group_holdout(
    df: pd.DataFrame,
    group_cols: list[str],
    weight_col: str,
    target_w: float,
    seed: int,
) -> tuple[np.ndarray, float]:
    """
    Assign whole groups to OOS until cumulative group-weight reaches/exceeds target_w.
    Returns:
      - mask_oos: boolean mask over df
      - achieved_w: achieved OOS weighted sum
    """
    if len(df) == 0:
        return np.zeros(0, dtype=bool), 0.0

    g = (
        df.groupby(group_cols, dropna=False)[weight_col]
          .sum()
          .reset_index(name="group_w")
    )

    total_w = float(g["group_w"].sum())
    if total_w <= 0:
        raise ValueError("Total weight <= 0. Check WP column.")

    rng = np.random.default_rng(seed)
    g = g.iloc[rng.permutation(len(g))].reset_index(drop=True)

    cum = g["group_w"].cumsum().to_numpy()
    idx = int(np.searchsorted(cum, target_w, side="left"))
    idx = min(idx, len(g) - 1)

    oos_key = g.loc[:idx, group_cols].drop_duplicates()

    marked = (
        df[group_cols]
        .merge(oos_key, on=group_cols, how="left", indicator=True)["_merge"]
        .eq("both")
        .to_numpy()
    )

    achieved_w = float(df.loc[marked, weight_col].sum())
    return marked, achieved_w

def balanced_oos_split_by_wave(
    df: pd.DataFrame,
    survey_col: str,
    weight_col: str,
    group_cols: list[str],
    target_oos_share: float,
    seed: int,
    waves=(2015, 2021),
):
    """
    Wave-balanced OOS by WP across waves.
    - If both waves present: target OOS weight split 50/50 across waves.
    - Grouping is done WITHIN each wave.
    Returns:
      - full_mask
      - achieved_total_w
      - target_total_oos_w
      - achieved_by_wave
    """
    df = df.copy().reset_index(drop=True)

    if len(df) == 0:
        return np.zeros(0, dtype=bool), 0.0, 0.0, {}

    df[survey_col] = pd.to_numeric(df[survey_col], errors="coerce")

    total_w = float(df[weight_col].sum())
    if total_w <= 0:
        raise ValueError("Total weight <= 0. Check WP column.")

    target_total_oos_w = target_oos_share * total_w
    present_waves = [w for w in waves if (df[survey_col] == w).any()]

    if len(present_waves) == 0:
        raise ValueError(f"No rows found for waves {waves} in column '{survey_col}'.")

    # One-wave case
    if len(present_waves) == 1:
        w = present_waves[0]
        pos = np.flatnonzero(df[survey_col].to_numpy() == w)
        df_w = df.iloc[pos].copy().reset_index(drop=True)

        mask_w, achieved_w = weighted_group_holdout(
            df=df_w,
            group_cols=group_cols,
            weight_col=weight_col,
            target_w=target_total_oos_w,
            seed=seed,
        )

        full_mask = np.zeros(len(df), dtype=bool)
        full_mask[pos] = mask_w
        return full_mask, achieved_w, target_total_oos_w, {w: achieved_w}

    # Two-wave case -> half target in each wave
    per_wave_target = target_total_oos_w / 2.0
    full_mask = np.zeros(len(df), dtype=bool)
    achieved_by_wave = {}

    for k, w in enumerate(present_waves[:2]):
        pos = np.flatnonzero(df[survey_col].to_numpy() == w)
        df_w = df.iloc[pos].copy().reset_index(drop=True)

        mask_w, achieved_w = weighted_group_holdout(
            df=df_w,
            group_cols=group_cols,
            weight_col=weight_col,
            target_w=per_wave_target,
            seed=seed + 1000 * k + int(w),
        )

        full_mask[pos] = mask_w
        achieved_by_wave[w] = achieved_w

    achieved_total = float(sum(achieved_by_wave.values()))
    return full_mask, achieved_total, target_total_oos_w, achieved_by_wave

def report_oos_wave_balance(
    df: pd.DataFrame,
    mask_oos: np.ndarray,
    survey_col: str,
    weight_col: str,
    waves=(2015, 2021),
):
    oos = df.loc[mask_oos].copy()
    by_wave = {}
    for w in waves:
        by_wave[w] = float(oos.loc[oos[survey_col] == w, weight_col].sum())
    return by_wave

def prepare_input_df(df0: pd.DataFrame) -> tuple[pd.DataFrame, str, str, str, str]:
    """
    Standardize and clean one destinations_* table.
    Returns:
      cleaned df, weight col, origin col, dest col, survey col
    """
    df0 = df0.copy()

    wcol = get_weight_col(df0)
    ocol = pick_col(df0, ORIGIN_COL_CANDIDATES, "origin")
    dcol = pick_col(df0, DEST_COL_CANDIDATES, "destination")
    pcol = pick_col(df0, PERSON_COL_CANDIDATES, "person (HHNR)")
    scol = pick_col(df0, SURVEY_COL_CANDIDATES, "survey/year")

    # numeric coercion
    df0[wcol] = pd.to_numeric(df0[wcol], errors="coerce")
    df0[ocol] = pd.to_numeric(df0[ocol], errors="coerce")
    df0[dcol] = pd.to_numeric(df0[dcol], errors="coerce")
    df0[pcol] = pd.to_numeric(df0[pcol], errors="coerce")
    df0[scol] = pd.to_numeric(df0[scol], errors="coerce")

    # keep valid rows only
    df0 = df0.dropna(subset=[wcol, ocol, dcol, pcol, scol]).copy()
    df0 = df0[df0[wcol] > 0].copy()

    # integerize IDs
    df0[ocol] = df0[ocol].astype(int)
    df0[dcol] = df0[dcol].astype(int)
    df0[pcol] = df0[pcol].astype(int)
    df0[scol] = df0[scol].astype(int)

    return df0, wcol, ocol, dcol, pcol, scol

def do_split_and_write(
    df: pd.DataFrame,
    seg: str,
    out_origin: str,
    out_person: str,
    wcol: str,
    ocol: str,
    dcol: str,
    pcol: str,
    scol: str,
):
    df = df.copy().reset_index(drop=True)

    # leak-proof person ID within wave
    df["person_uid"] = df[pcol].astype(str) + "_" + df[scol].astype(str)

    total_w = float(df[wcol].sum())
    print(f"Total rows={len(df):,} | Total WP sum={total_w:.2f}")

    # --------------------------------------------------------
    # byOrigin (GLOBAL leave-origins-out)
    # --------------------------------------------------------
    target_oos_w = TARGET_OOS_SHARE * total_w

    mask_oos_origin, achieved_oos_w_origin = weighted_group_holdout(
        df=df,
        group_cols=[ocol],
        weight_col=wcol,
        target_w=target_oos_w,
        seed=SEED_ORIGIN,
    )

    oos_origin = df.loc[mask_oos_origin].copy()
    train_origin = df.loc[~mask_oos_origin].copy()

    achieved_share_origin = float(
        oos_origin[wcol].sum() /
        (train_origin[wcol].sum() + oos_origin[wcol].sum() + 1e-12)
    )
    by_wave_origin = report_oos_wave_balance(
        df=df,
        mask_oos=mask_oos_origin,
        survey_col=scol,
        weight_col=wcol,
        waves=(2015, 2021),
    )

    print(
        f"[byOrigin GLOBAL] OOS weighted share achieved: {achieved_share_origin:.3f} "
        f"(target {TARGET_OOS_SHARE:.2f}) | "
        f"target_oos_w={target_oos_w:.2f} achieved_oos_w={achieved_oos_w_origin:.2f}"
    )
    print(
        "[byOrigin GLOBAL] OOS by wave (WP): " +
        ", ".join([f"{k}: {v:.2f}" for k, v in by_wave_origin.items()])
    )

    write_split(
        out_origin,
        seg,
        train_origin.drop(columns=["person_uid"], errors="ignore"),
        oos_origin.drop(columns=["person_uid"], errors="ignore"),
    )

    # --------------------------------------------------------
    # byPerson (wave-balanced)
    # --------------------------------------------------------
    mask_oos_person, achieved_oos_w_person, target_oos_w_person, by_wave_person = balanced_oos_split_by_wave(
        df=df,
        survey_col=scol,
        weight_col=wcol,
        group_cols=["person_uid"],
        target_oos_share=TARGET_OOS_SHARE,
        seed=SEED_PERSON,
        waves=(2015, 2021),
    )

    oos_person = df.loc[mask_oos_person].copy()
    train_person = df.loc[~mask_oos_person].copy()

    achieved_share_person = float(
        oos_person[wcol].sum() /
        (train_person[wcol].sum() + oos_person[wcol].sum() + 1e-12)
    )

    print(
        f"[byPerson] OOS weighted share achieved: {achieved_share_person:.3f} "
        f"(target {TARGET_OOS_SHARE:.2f}) | "
        f"target_oos_w={target_oos_w_person:.2f} achieved_oos_w={achieved_oos_w_person:.2f}"
    )
    print(
        "[byPerson] OOS by wave (WP): " +
        ", ".join([f"{k}: {v:.2f}" for k, v in by_wave_person.items()])
    )

    write_split(
        out_person,
        seg,
        train_person.drop(columns=["person_uid"], errors="ignore"),
        oos_person.drop(columns=["person_uid"], errors="ignore"),
    )

# -----------------------------
# RUN
# -----------------------------
files = sorted(glob.glob(INPUT_GLOB))
if not files:
    raise FileNotFoundError(f"No files found matching: {INPUT_GLOB}")

print(f"Found {len(files)} destination files:")
for f in files:
    print(" -", f)

for fpath in files:
    seg = segment_from_filename(fpath)
    print(f"\n=== Processing segment {seg} ({fpath}) ===")

    df_raw = pd.read_csv(fpath, low_memory=False)
    df0, wcol, ocol, dcol, pcol, scol = prepare_input_df(df_raw)

    # -------------------------
    # Base export
    # -------------------------
    print("\n--- BASE export (with Liechtenstein) ---")
    do_split_and_write(
        df=df0,
        seg=seg,
        out_origin=OUT_DIR_ORIGIN,
        out_person=OUT_DIR_PERSON,
        wcol=wcol,
        ocol=ocol,
        dcol=dcol,
        pcol=pcol,
        scol=scol,
    )

    # -------------------------
    # woLIE export
    # -------------------------
    df_wo = df0[
        (~df0[ocol].isin(LIE_NPVM)) &
        (~df0[dcol].isin(LIE_NPVM))
    ].copy().reset_index(drop=True)

    print("\n--- woLIE export (orig/dest not in Liechtenstein) ---")
    print(f"[woLIE] filtered rows: {len(df_wo):,} (removed {len(df0) - len(df_wo):,})")

    do_split_and_write(
        df=df_wo,
        seg=seg,
        out_origin=OUT_DIR_ORIGIN_WO,
        out_person=OUT_DIR_PERSON_WO,
        wcol=wcol,
        ocol=ocol,
        dcol=dcol,
        pcol=pcol,
        scol=scol,
    )

print("\n[DONE] Rebuilt all train/oos splits from the corrected destination tables.")

## K selection

In [ ]:
# ============================================================
# K-CHOOSING EVALUATION
# - Uses ONLY the external TRAIN dataset: Trips/byPerson/*_train.csv
# - Creates an INNER weighted 80/20 split on that training data
# - Evaluates SAME MNL model for K in [100,300,500,800,1000,2000,3000]
# - Uniform sampling of non-chosen alternatives
# - Outputs saved in: ModelRuns/K_chosing
#
# INNER SPLIT:
#   - random
#   - weighted by WP
#   - NO byPerson grouping (as requested)
#
# NOTE:
#   - This script is for HYPERPARAMETER CHOOSING only.
#   - After selecting K, you should re-estimate the final model on the FULL
#     external 80% train sample, then evaluate once on the external 20% OOS.
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import torch
import openmatrix as omx
from pathlib import Path
from math import erf, sqrt

# -----------------------------
# CONFIG
# -----------------------------
TRIPS_DIR = "Trips/byPerson"   # uses only destinations_{SEG}_train.csv
SEGMENTS  = ["YS", "OS", "YL", "OL"]

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX  = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT   = "utility_emu"
MAP_NAME  = "NO"

OUT_DIR = "ModelRuns/K_chosing"
os.makedirs(OUT_DIR, exist_ok=True)

K_LIST = [100, 300, 500, 800, 1000, 2000, 3000]

# external-train -> inner split
INNER_VAL_SHARE = 0.20
SEED_SPLIT = 321
SEED_SAMPLE = 123

# Torch
DEVICE = "cpu"
DTYPE  = torch.float32

# Optimization
EPOCHS = 30
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# FULL-choice-set evaluation subset sizes on INNER-VAL
# set None if you want all rows, but runtime can explode
FULL_EVAL_MAX = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
FULL_EVAL_SEED = 777
FULL_EVAL_CHUNK = 256

LOG1P_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]

# -----------------------------
# Helpers
# -----------------------------
def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

# -----------------------------
# IO helpers
# -----------------------------
def load_segment_csv(seg: str, split: str = "train") -> pd.DataFrame:
    path = os.path.join(TRIPS_DIR, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    need = [ORIG_COL, DEST_COL, WP_COL]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()
    return df.reset_index(drop=True)

def load_emu_matrix_and_mapping(omx_path: Path, emu_mat: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if emu_mat not in f.list_matrices():
        raise ValueError(f"Matrix '{emu_mat}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        no = np.asarray(m[:]).astype(int)   # idx -> zone_id
    except Exception:
        no = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(no)}

    M = f[emu_mat]
    print(f"[OMX] Loading {emu_mat} into RAM... shape={M.shape} (float32)")
    emu = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return emu, zone_to_idx, no

# -----------------------------
# Weighted inner split (row-level)
# -----------------------------
def weighted_row_holdout_split(
    df: pd.DataFrame,
    weight_col: str,
    val_share: float,
    seed: int,
):
    """
    Random row-level split such that validation reaches approximately val_share
    in weighted terms.
    """
    if len(df) == 0:
        return df.copy(), df.copy()

    total_w = float(df[weight_col].sum())
    if total_w <= 0:
        raise ValueError("Total weight <= 0")

    target_val_w = val_share * total_w

    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    dfp = df.iloc[order].copy().reset_index(drop=True)

    cum_w = dfp[weight_col].cumsum().to_numpy()
    idx = int(np.searchsorted(cum_w, target_val_w, side="left"))
    idx = min(idx, len(dfp) - 1)

    val_df = dfp.iloc[:idx+1].copy().reset_index(drop=True)
    train_df = dfp.iloc[idx+1:].copy().reset_index(drop=True)

    return train_df, val_df

# -----------------------------
# Preprocess helpers
# -----------------------------
def standardize_global_allzones(
    tz_feat: pd.DataFrame,
    cols: list[str],
    suffix: str = "_z"
):
    """
    GLOBAL z-score computed once on ALL zones in tz_feat.
    This keeps the same logic as your previous script.
    """
    X = tz_feat[cols].replace([np.inf, -np.inf], np.nan)

    mu = X.mean(axis=0, skipna=True)
    sd = X.std(axis=0, skipna=True).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    if tz_feat[z_cols].isna().any().any():
        n_na = int(tz_feat[z_cols].isna().sum().sum())
        print(f"[WARN] z-score produced {n_na} NaNs (likely from all-NaN col or sd=0).")

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)

    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts

    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy().reset_index(drop=True)
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)  # chosen always first

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
        dest_set[good],
    )

# -----------------------------
# Model: MNL train/eval on sampled sets
# -----------------------------
def train_mnl(emu_set, X_set, y, w, epochs=30, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, J = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]

            loss = -(w_b * ll).sum() / (w_b.sum() + 1e-9)

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu()) * float(w_b.sum().detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    alpha_hat, beta_hat = best_state
    return alpha_hat, beta_hat

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)

        chosen_logp = logP[:, 0]
        sum_w = float(w.sum().cpu())

        nll = float((-(w * chosen_logp)).sum().cpu() / (sum_w + 1e-9))

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))
        rank_median = float(torch.median(rank).cpu())

        top1   = float((w * (rank <= 1).float()).sum().cpu() / (sum_w + 1e-9))
        top10  = float((w * (rank <= 10).float()).sum().cpu() / (sum_w + 1e-9))
        top50  = float((w * (rank <= 50).float()).sum().cpu() / (sum_w + 1e-9))
        top100 = float((w * (rank <= 100).float()).sum().cpu() / (sum_w + 1e-9))

        ll_model = float((w * chosen_logp).sum().cpu())
        J = V.shape[1]
        ll0 = sum_w * float(np.log(1.0 / J))
        r2 = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    return {
        "NLL": nll,
        "MRR": mrr,
        "McFadden_R2": float(r2),
        "rank_median": rank_median,
        "Top1": top1,
        "Top10": top10,
        "Top50": top50,
        "Top100": top100,
        "LL_model": ll_model,
        "LL0_uniform": ll0,
        "J": int(emu_set.shape[1]),
    }

def sampled_shares_metrics(
    df_obs: pd.DataFrame,
    dest_set: np.ndarray,
    emu_set: torch.Tensor,
    X_set: torch.Tensor,
    w: torch.Tensor,
    alpha: float,
    beta: np.ndarray,
):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        P = torch.softmax(V, dim=1).detach().cpu().numpy()

    w_np = w.detach().cpu().numpy().astype(np.float64)
    chosen = df_obs[DEST_COL].to_numpy(dtype=int)

    obs_mass = pd.Series(w_np).groupby(chosen).sum()

    pred_mass = {}
    for n in range(dest_set.shape[0]):
        wn = float(w_np[n])
        for j in range(dest_set.shape[1]):
            zid = int(dest_set[n, j])
            pred_mass[zid] = pred_mass.get(zid, 0.0) + wn * float(P[n, j])

    pred_mass = pd.Series(pred_mass)

    all_ids = obs_mass.index.union(pred_mass.index)
    obs = obs_mass.reindex(all_ids, fill_value=0.0).to_numpy(dtype=np.float64)
    pred = pred_mass.reindex(all_ids, fill_value=0.0).to_numpy(dtype=np.float64)

    obs = obs / (obs.sum() + 1e-12)
    pred = pred / (pred.sum() + 1e-12)

    spear = float(pd.Series(obs).corr(pd.Series(pred), method="spearman")) if len(obs) > 2 else np.nan
    js = js_div(obs, pred)
    return spear, js

# -----------------------------
# FULL-choice-set evaluation on INNER-VAL subset
# -----------------------------
def full_choice_eval_subset(
    eval_df: pd.DataFrame,
    tz_feat_seg: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
):
    if (max_n is not None) and (len(eval_df) > max_n):
        df = eval_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = eval_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w = df[WP_COL].to_numpy(dtype=np.float64)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)
    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w = w[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat_seg["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat_seg[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}
    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)

    obs_mass = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    mrr_num = 0.0
    top1_num = top10_num = top50_num = top100_num = 0.0
    ranks = np.empty(len(df), dtype=np.int64)
    ll_model = 0.0

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)

            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

        nll_num += wn * (-logp_chosen)
        ll_model += wn * logp_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))

        top1_num   += wn * (1.0 if rank <= 1 else 0.0)
        top10_num  += wn * (1.0 if rank <= 10 else 0.0)
        top50_num  += wn * (1.0 if rank <= 50 else 0.0)
        top100_num += wn * (1.0 if rank <= 100 else 0.0)

    nll_full = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)
    share_spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    share_js = js_div(S_obs, S_pred)

    return {
        "NLL_full": nll_full,
        "MRR_full": mrr_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "Top1_full": float(top1_num / (sum_w + 1e-9)),
        "Top10_full": float(top10_num / (sum_w + 1e-9)),
        "Top50_full": float(top50_num / (sum_w + 1e-9)),
        "Top100_full": float(top100_num / (sum_w + 1e-9)),
        "share_spearman_full": share_spear,
        "share_JS_full": float(share_js),
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }

# -----------------------------
# Plot helpers
# -----------------------------
def save_metric_plot(df, metric, out_path, title):
    plt.figure(figsize=(9, 5))
    for seg in SEGMENTS:
        d = df[df["segment"] == seg].sort_values("K")
        if len(d) == 0:
            continue
        plt.plot(d["K"], d[metric], marker="o", label=seg)
    plt.xlabel("K")
    plt.ylabel(metric)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()

# ============================================================
# MAIN
# ============================================================

# Load TZ features
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
need_cols = ["npvm_id"] + LOG1P_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

tz_feat0 = tz[["npvm_id"] + LOG1P_COLS + ["geometry"]].copy()
tz_feat0["npvm_id"] = pd.to_numeric(tz_feat0["npvm_id"], errors="coerce")
tz_feat0 = tz_feat0.dropna(subset=["npvm_id"]).copy()
tz_feat0["npvm_id"] = tz_feat0["npvm_id"].astype(int)

# Global z-score once, same logic as your previous script
tz_feat0, z_cols, mu_all, sd_all = standardize_global_allzones(tz_feat0, LOG1P_COLS, suffix="_z")
print("[Z] Global scaling computed on ALL TZ zones.")

# Load EMU matrix + mapping
emu_mat, zone_to_idx, idx_to_zone = load_emu_matrix_and_mapping(UTIL_OMX, EMU_MAT, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

results = []

for K in K_LIST:
    print("\n" + "#" * 110)
    print(f"# K = {K}")
    print("#" * 110)

    for seg in SEGMENTS:
        print("\n" + "=" * 95)
        print(f"Segment {seg} | K={K}")
        print("=" * 95)

        # use ONLY the external-train sample
        ext_train_df = load_segment_csv(seg, "train")

        # inner weighted split on training only
        inner_train_df, inner_val_df = weighted_row_holdout_split(
            df=ext_train_df,
            weight_col=WP_COL,
            val_share=INNER_VAL_SHARE,
            seed=SEED_SPLIT + K + hash(seg) % 1000
        )

        wp_total = ext_train_df[WP_COL].sum()
        wp_inner_train = inner_train_df[WP_COL].sum()
        wp_inner_val = inner_val_df[WP_COL].sum()

        print(f"[INNER SPLIT] external-train rows={len(ext_train_df):,} | WP={wp_total:.2f}")
        print(f"[INNER SPLIT] inner-train rows={len(inner_train_df):,} | WP={wp_inner_train:.2f}")
        print(f"[INNER SPLIT] inner-val   rows={len(inner_val_df):,} | WP={wp_inner_val:.2f}")
        print(f"[INNER SPLIT] inner-val weighted share={wp_inner_val/(wp_total+1e-12):.3f} (target {INNER_VAL_SHARE:.2f})")

        tz_feat_seg = tz_feat0

        # Build sampled designs
        emu_tr, X_tr, y_tr, w_tr, inner_train_df2, destset_tr = build_design(
            inner_train_df, tz_feat_seg, z_cols, emu_mat, zone_to_idx, k=K, seed=SEED_SAMPLE + 10 + K
        )
        emu_va, X_va, y_va, w_va, inner_val_df2, destset_va = build_design(
            inner_val_df, tz_feat_seg, z_cols, emu_mat, zone_to_idx, k=K, seed=SEED_SAMPLE + 20 + K
        )

        print(f"[DATA] inner-train rows={len(inner_train_df2):,} | inner-val rows={len(inner_val_df2):,}")
        print(f"[DATA] inner-train WP={inner_train_df2[WP_COL].sum():.2f} | inner-val WP={inner_val_df2[WP_COL].sum():.2f}")

        # Train on INNER-TRAIN only
        alpha_hat, beta_hat = train_mnl(
            emu_tr, X_tr, y_tr, w_tr,
            epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
        )

        # Sampled metrics
        met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat)
        met_va = eval_sampled_metrics(emu_va, X_va, w_va, alpha_hat, beta_hat)

        # Sampled shares
        share_spear_tr, share_js_tr = sampled_shares_metrics(
            df_obs=inner_train_df2, dest_set=destset_tr,
            emu_set=emu_tr, X_set=X_tr, w=w_tr,
            alpha=alpha_hat, beta=beta_hat
        )
        share_spear_va, share_js_va = sampled_shares_metrics(
            df_obs=inner_val_df2, dest_set=destset_va,
            emu_set=emu_va, X_set=X_va, w=w_va,
            alpha=alpha_hat, beta=beta_hat
        )

        # Full-choice-set evaluation on INNER-VAL subset
        max_n = FULL_EVAL_MAX.get(seg, 8000)
        full_met = full_choice_eval_subset(
            eval_df=inner_val_df2,
            tz_feat_seg=tz_feat_seg,
            z_cols=z_cols,
            emu_mat=emu_mat,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            alpha_hat=alpha_hat,
            beta_hat=beta_hat,
            max_n=max_n,
            seed=FULL_EVAL_SEED + (hash(seg) % 1000) + K,
            chunk=FULL_EVAL_CHUNK,
        )

        row = {
            "K": K,
            "segment": seg,
            "alpha_emu": float(alpha_hat),

            "N_external_train": int(len(ext_train_df)),
            "WP_external_train": float(wp_total),

            "N_inner_train": int(len(inner_train_df2)),
            "WP_inner_train": float(inner_train_df2[WP_COL].sum()),
            "N_inner_val": int(len(inner_val_df2)),
            "WP_inner_val": float(inner_val_df2[WP_COL].sum()),
            "inner_val_share_weighted": float(inner_val_df2[WP_COL].sum() / (wp_total + 1e-12)),

            "NLL_inner_train": met_tr["NLL"],
            "MRR_inner_train": met_tr["MRR"],
            "McFadden_R2_inner_train": met_tr["McFadden_R2"],
            "rank_median_inner_train": met_tr["rank_median"],
            "Top1_inner_train": met_tr["Top1"],
            "Top10_inner_train": met_tr["Top10"],
            "Top50_inner_train": met_tr["Top50"],
            "Top100_inner_train": met_tr["Top100"],
            "share_spearman_inner_train": float(share_spear_tr),
            "share_JS_inner_train": float(share_js_tr),

            "NLL_inner_val": met_va["NLL"],
            "MRR_inner_val": met_va["MRR"],
            "McFadden_R2_inner_val": met_va["McFadden_R2"],
            "rank_median_inner_val": met_va["rank_median"],
            "Top1_inner_val": met_va["Top1"],
            "Top10_inner_val": met_va["Top10"],
            "Top50_inner_val": met_va["Top50"],
            "Top100_inner_val": met_va["Top100"],
            "share_spearman_inner_val": float(share_spear_va),
            "share_JS_inner_val": float(share_js_va),
        }
        row.update(full_met)
        results.append(row)

        print("[SAMPLED INNER-TRAIN] "
              f"NLL={row['NLL_inner_train']:.4f} | MRR={row['MRR_inner_train']:.4f} | "
              f"Top1={row['Top1_inner_train']:.3f} Top10={row['Top10_inner_train']:.3f} "
              f"Top50={row['Top50_inner_train']:.3f} Top100={row['Top100_inner_train']:.3f} | "
              f"shareSp={row['share_spearman_inner_train']:.3f} JS={row['share_JS_inner_train']:.3f}")

        print("[SAMPLED INNER-VAL  ] "
              f"NLL={row['NLL_inner_val']:.4f} | MRR={row['MRR_inner_val']:.4f} | "
              f"Top1={row['Top1_inner_val']:.3f} Top10={row['Top10_inner_val']:.3f} "
              f"Top50={row['Top50_inner_val']:.3f} Top100={row['Top100_inner_val']:.3f} | "
              f"shareSp={row['share_spearman_inner_val']:.3f} JS={row['share_JS_inner_val']:.3f}")

        if full_met:
            print("[FULL INNER-VAL SUB] "
                  f"NLL_full={row['NLL_full']:.4f} | MRR_full={row['MRR_full']:.4f} | "
                  f"Top1={row['Top1_full']:.3f} Top10={row['Top10_full']:.3f} "
                  f"Top50={row['Top50_full']:.3f} Top100={row['Top100_full']:.3f} | "
                  f"shareSp={row['share_spearman_full']:.3f} JS={row['share_JS_full']:.3f} | "
                  f"N_full={row.get('N_full_eval', np.nan)} J={row.get('J_full', np.nan)}")

# Save CSV
res_df = pd.DataFrame(results).sort_values(["K", "segment"]).reset_index(drop=True)
csv_path = os.path.join(OUT_DIR, "k_results.csv")
res_df.to_csv(csv_path, index=False)
print(f"\n[OK] wrote {csv_path}")

# -----------------------------
# Save plots
# -----------------------------
save_metric_plot(
    res_df, "NLL_inner_val",
    os.path.join(OUT_DIR, "plot_NLL_inner_val.png"),
    "K choosing - sampled NLL on inner validation"
)

save_metric_plot(
    res_df, "MRR_inner_val",
    os.path.join(OUT_DIR, "plot_MRR_inner_val.png"),
    "K choosing - sampled MRR on inner validation"
)

save_metric_plot(
    res_df, "Top10_inner_val",
    os.path.join(OUT_DIR, "plot_Top10_inner_val.png"),
    "K choosing - sampled Top10 on inner validation"
)

save_metric_plot(
    res_df, "NLL_full",
    os.path.join(OUT_DIR, "plot_NLL_full_inner_val.png"),
    "K choosing - full-choice-set NLL on inner validation subset"
)

save_metric_plot(
    res_df, "MRR_full",
    os.path.join(OUT_DIR, "plot_MRR_full_inner_val.png"),
    "K choosing - full-choice-set MRR on inner validation subset"
)

print(f"\n[DONE] Plots saved in: {OUT_DIR}")

## Proposed specification

In [ ]:
# ============================================================
# BASELINE ESTIMATION + FULL REPORT + VCOV + COEFFICIENT TESTS
# (withLIE, byPerson, log1p, global z, EMU)
#
# ADDITIONS IN THIS VERSION:
#   - computes full Hessian-based approximate variance-covariance matrix per segment
#   - exports vcov matrix per segment
#   - exports coefficient table as before
#   - exports tests of coefficient differences:
#       * across segments, same parameter
#       * within segment, pairwise parameter differences
#
# IMPORTANT:
#   - coefficients are STILL estimated exactly as before by weighted MNL training
#   - Hessian is only used post-estimation for approximate inference
#   - across-segment difference tests assume segment-specific estimates are independent
# ============================================================

import os
import math
import random
import itertools
import numpy as np
import pandas as pd
import geopandas as gpd
import tables as tb

import torch
import openmatrix as omx
from pathlib import Path

import matplotlib.pyplot as plt


# -----------------------------
# CONFIG (withLIE + byPerson)
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"
DIST_COL  = "dist_km"

TRIPS_DIR = "Trips/byPerson"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")

UTIL_OMX      = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME  = "utility_emu"

DIST_OMX      = READY_DIR / "distance_avg_2023_ready.omx"
DIST_MAT_NAME = "avg_distances_ready"

MAP_NAME   = "NO"

OUT_DIR = "ModelRuns/BaselineFinal2"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DIST_OOS_DIR = os.path.join(OUT_DIR, "distance_oos_triplevel")
os.makedirs(DIST_OOS_DIR, exist_ok=True)

RESULT_DIR = os.path.join(OUT_DIR, "Result")
os.makedirs(RESULT_DIR, exist_ok=True)

# NEW OUTPUTS
VCOV_DIR = os.path.join(OUT_DIR, "vcov")
os.makedirs(VCOV_DIR, exist_ok=True)

TEST_DIR = os.path.join(OUT_DIR, "coef_tests")
os.makedirs(TEST_DIR, exist_ok=True)

ATTR_CSV_OUT  = os.path.join(RESULT_DIR, "Attractivity.csv")
ATTR_GPKG_OUT = os.path.join(RESULT_DIR, "Attractivity.gpkg")
UTIL_SEG_OMX  = os.path.join(RESULT_DIR, "utilities_by_segment.omx")

K = 1000
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

FULL_EVAL_MAX = {
    "YS": 20000,
    "OS": 8000,
    "YL": 8000,
    "OL": 8000,
}
FULL_EVAL_SEED  = 777
FULL_EVAL_CHUNK = 256

SE_MAX_TRAIN = {
    "YS": 100000,
    "OS": 100000,
    "YL": 100000,
    "OL": 100000,
}
SE_SEED = 2024

BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

TOPKS_SAMPLED = (5, 10)
TOPKS_FULL    = (5, 10)

BASE_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]

PARAM_NAMES = ["alpha_emu"] + BASE_COLS


# ============================================================
# Global reproducibility helper
# ============================================================
def set_all_seeds(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Small helpers
# ============================================================
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300); q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def p_value_two_sided_z(z):
    return 2.0 * (1.0 - norm_cdf(abs(float(z))))

def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.1:
        return "."
    return ""

def safe_sqrt(x):
    return float(np.sqrt(max(float(x), 0.0)))


# ============================================================
# IO helpers
# ============================================================
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()

    if DIST_COL in df.columns:
        df[DIST_COL] = pd.to_numeric(df[DIST_COL], errors="coerce")

    return df

def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone


# ============================================================
# Preprocess helpers
# ============================================================
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd

def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0]  = c
        dest_set[i, 1:] = alts
    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]
    dest_set2 = dest_set[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
        dest_set2,
        orig_idx2,
        dest_idx2,
    )


# ============================================================
# Model
# ============================================================
def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)

            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum()

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted-avg NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(5,10), want_ff=True):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        P = torch.exp(logP)

        chosen_logp = logP[:, 0]
        chosen_p    = P[:, 0]
        sum_w = float(w.sum().cpu())

        nll_sum = float((-(w * chosen_logp)).sum().cpu())
        nll_avg = nll_sum / (sum_w + 1e-9)

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))

        out = {
            "NLL_sum": nll_sum,
            "NLL_avg": nll_avg,
            "MRR": mrr,
            "sum_WP": sum_w,
            "N": int(V.shape[0]),
            "J": int(V.shape[1]),
        }

        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))

        if want_ff:
            out["FF"] = float((w * chosen_p).sum().cpu() / (sum_w + 1e-9))

    return out


# ============================================================
# FULL-choice-set evaluation on an OOS subset
# ============================================================
def full_eval_and_shares_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    dist_mat: np.ndarray | None,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(5,10),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w    = df[WP_COL].to_numpy(dtype=np.float64)

    has_dist = (DIST_COL in df.columns) and df[DIST_COL].notna().any()
    obs_dist = df[DIST_COL].to_numpy(dtype=np.float64) if has_dist else None

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    if has_dist:
        good = good & np.isfinite(obs_dist) & (obs_dist >= 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w        = w[good]
    if has_dist:
        obs_dist = obs_dist[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}, None, (None, None), None

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass  = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    ll_model = 0.0
    mrr_num = 0.0
    ff_num = 0.0
    top_num = {k: 0.0 for k in topKs}
    ranks   = np.empty(len(df), dtype=np.int64)

    exp_dist = np.zeros(len(df), dtype=np.float64) if (dist_mat is not None and has_dist) else None

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse
        p_chosen = math.exp(logp_chosen)

        nll_num  += wn * (-logp_chosen)
        ll_model += wn * (logp_chosen)
        ff_num   += wn * p_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        if exp_dist is not None:
            expd = 0.0

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

            if exp_dist is not None:
                dist_blk = dist_mat[o, j0:j1].astype(np.float64)
                expd += float(np.sum(Pblk * dist_blk))

        if exp_dist is not None:
            exp_dist[n] = expd

    nll_full = float(nll_num)
    nll_full_avg = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    ff_full = float(ff_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs  = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)

    pear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="pearson")) if len(S_obs) > 2 else np.nan
    spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    mae  = float(np.mean(np.abs(S_obs - S_pred)))
    rmse = float(np.sqrt(np.mean((S_obs - S_pred) ** 2)))
    js   = float(js_div(S_obs, S_pred))

    dist_summary = {}
    if exp_dist is not None:
        wnorm = w / (sum_w + 1e-12)
        obs_mean = float(np.sum(wnorm * obs_dist))
        pred_mean = float(np.sum(wnorm * exp_dist))
        dist_summary = {
            "dist_obs_mean_km": obs_mean,
            "dist_pred_mean_km": pred_mean,
            "dist_mean_diff_km": float(pred_mean - obs_mean),
            "dist_obs_p50_km": float(np.quantile(obs_dist, 0.50)),
            "dist_pred_p50_km": float(np.quantile(exp_dist, 0.50)),
            "dist_obs_p90_km": float(np.quantile(obs_dist, 0.90)),
            "dist_pred_p90_km": float(np.quantile(exp_dist, 0.90)),
        }

    out = {
        "NLL_full_sum": nll_full,
        "NLL_full_avg": nll_full_avg,
        "MRR_full": mrr_full,
        "FF_full": ff_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "Pearson_share": pear,
        "Spearman_share": spear,
        "MAE_share": mae,
        "RMSE_share": rmse,
        "JS_share": js,
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    out.update(dist_summary)

    trip_distance_df = None
    if exp_dist is not None:
        trip_distance_df = pd.DataFrame({
            "orig_zone": df[ORIG_COL].to_numpy(dtype=int),
            "dest_zone_obs": df[DEST_COL].to_numpy(dtype=int),
            "WP": w.astype(np.float64),
            "dist_obs_km": obs_dist.astype(np.float64),
            "dist_exp_km": exp_dist.astype(np.float64),
        })

    return out, {
        "tz_ids": all_zone_ids,
        "S_obs": S_obs,
        "S_pred": S_pred,
        "Aj": Aj.astype(np.float64),
        "orig_idx": orig_idx,
        "w": w,
    }, (obs_dist, exp_dist), trip_distance_df


# ============================================================
# Hessian-based approximate inference
# ============================================================
def approx_hessian_inference(
    emu_set: torch.Tensor,
    X_set: torch.Tensor,
    y: torch.Tensor,
    w: torch.Tensor,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int,
    seed: int,
    ridge: float = 1e-8,
):
    """
    Returns:
      vcov_sum : approximate covariance matrix of theta_hat
      se       : sqrt(diag(vcov_sum))
      H_sum    : Hessian of weighted SUM objective
      used_n   : number of observations used
      used_sum_w : total weight used in Hessian approximation

    theta = [alpha_emu, beta_1, ..., beta_P]
    """
    N = emu_set.shape[0]
    rng = np.random.default_rng(seed)

    if N > max_n:
        idx = rng.choice(N, size=max_n, replace=False)
        idx = torch.tensor(idx, dtype=torch.long, device=DEVICE)
        emu = emu_set[idx]
        X   = X_set[idx]
        yy  = y[idx]
        ww  = w[idx]
    else:
        emu, X, yy, ww = emu_set, X_set, y, w

    used_n = int(emu.shape[0])
    used_sum_w = float(ww.sum().detach().cpu())

    Pdim = X.shape[2]

    alpha = torch.tensor([alpha_hat], dtype=DTYPE, device=DEVICE, requires_grad=True)
    beta  = torch.tensor(beta_hat, dtype=DTYPE, device=DEVICE, requires_grad=True)

    def loss_avg(a, b):
        V = a * emu + torch.einsum("bjp,p->bj", X, b)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        ll = logP[torch.arange(V.shape[0], device=DEVICE), yy]
        return (-(ww * ll)).sum() / (ww.sum() + 1e-12)

    L = loss_avg(alpha, beta)
    g = torch.autograd.grad(L, [alpha, beta], create_graph=True)
    g_vec = torch.cat([g[0].reshape(1), g[1].reshape(-1)], dim=0)

    H_avg = torch.zeros((1 + Pdim, 1 + Pdim), dtype=torch.float64, device=DEVICE)
    for i in range(1 + Pdim):
        gi = g_vec[i]
        hi = torch.autograd.grad(gi, [alpha, beta], retain_graph=True)
        hi_vec = torch.cat([hi[0].reshape(1), hi[1].reshape(-1)], dim=0)
        H_avg[i, :] = hi_vec.detach().to(torch.float64)

    H_avg = H_avg.cpu().numpy()
    H_sum = used_sum_w * H_avg

    H_sum = 0.5 * (H_sum + H_sum.T)
    H_sum = H_sum + ridge * np.eye(H_sum.shape[0], dtype=np.float64)

    vcov = np.linalg.pinv(H_sum)
    vcov = 0.5 * (vcov + vcov.T)

    se = np.sqrt(np.clip(np.diag(vcov), 0.0, np.inf))
    return vcov, se, H_sum, used_n, used_sum_w


# ============================================================
# Coefficient-difference tests
# ============================================================
def build_within_segment_diff_tests(segment, params, coef_vec, vcov):
    rows = []
    for i in range(len(params)):
        for j in range(i + 1, len(params)):
            p1 = params[i]
            p2 = params[j]
            delta = float(coef_vec[i] - coef_vec[j])
            var_delta = float(vcov[i, i] + vcov[j, j] - 2.0 * vcov[i, j])
            se_delta = safe_sqrt(var_delta)
            z_delta = delta / (se_delta + 1e-12)
            p_delta = p_value_two_sided_z(z_delta)
            rows.append({
                "segment": segment,
                "param_1": p1,
                "param_2": p2,
                "coef_1": float(coef_vec[i]),
                "coef_2": float(coef_vec[j]),
                "delta_1_minus_2": delta,
                "se_delta": se_delta,
                "z_delta": float(z_delta),
                "p_delta": float(p_delta),
                "stars_delta": stars(p_delta),
            })
    return rows

def build_across_segment_sameparam_tests(seg_to_coef, seg_to_vcov, params):
    rows = []
    seg_pairs = list(itertools.combinations(sorted(seg_to_coef.keys()), 2))
    for param_idx, param in enumerate(params):
        for s1, s2 in seg_pairs:
            b1 = float(seg_to_coef[s1][param_idx])
            b2 = float(seg_to_coef[s2][param_idx])

            # assuming independence across segment-specific estimates
            var_delta = float(seg_to_vcov[s1][param_idx, param_idx] + seg_to_vcov[s2][param_idx, param_idx])
            se_delta = safe_sqrt(var_delta)
            delta = b1 - b2
            z_delta = delta / (se_delta + 1e-12)
            p_delta = p_value_two_sided_z(z_delta)

            rows.append({
                "param": param,
                "segment_1": s1,
                "segment_2": s2,
                "coef_1": b1,
                "coef_2": b2,
                "delta_1_minus_2": float(delta),
                "se_delta": se_delta,
                "z_delta": float(z_delta),
                "p_delta": float(p_delta),
                "stars_delta": stars(p_delta),
                "assumption": "independent_segment_estimates",
            })
    return rows


# ============================================================
# Destination-level summaries and plots
# ============================================================
def incoming_accessibility_logsum(
    tz_ids: np.ndarray,
    tz_pos: dict,
    emu_mat: np.ndarray,
    alpha: float,
    orig_idx: np.ndarray,
    w: np.ndarray,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    chunk: int = 256,
):
    Msize = emu_mat.shape[0]
    w_by_o = np.zeros(Msize, dtype=np.float64)
    for o, wn in zip(orig_idx, w):
        w_by_o[int(o)] += float(wn)

    origins = np.where(w_by_o > 0)[0]
    w_o = w_by_o[origins]
    w_o = w_o / (w_o.sum() + 1e-12)

    L = np.full(len(tz_ids), np.nan, dtype=np.float64)

    tz_omx = np.array([zone_to_idx.get(int(z), -1) for z in tz_ids], dtype=int)
    good = (tz_omx >= 0)
    tz_omx_good = tz_omx[good]

    good_pos = np.where(good)[0]
    for local_idx, j in enumerate(tz_omx_good):
        vals = alpha * emu_mat[origins, j].astype(np.float64)
        m = np.max(vals)
        s = np.sum(w_o * np.exp(vals - m))
        L[good_pos[local_idx]] = float(m + np.log(s + 1e-300))

    return L

def plot_distance_plausibility(obs_dist, exp_dist, seg, out_png):
    obs_dist = np.asarray(obs_dist, dtype=np.float64)
    exp_dist = np.asarray(exp_dist, dtype=np.float64)

    finite = np.isfinite(obs_dist) & np.isfinite(exp_dist)
    obs_dist = obs_dist[finite]
    exp_dist = exp_dist[finite]

    if len(obs_dist) == 0:
        return

    obs_mean = float(np.mean(obs_dist))
    exp_mean = float(np.mean(exp_dist))

    xmax = float(max(np.max(obs_dist), np.max(exp_dist)))
    bins = np.linspace(0.0, xmax, 51)

    plt.figure()
    plt.hist(obs_dist, bins=bins, density=True, alpha=0.6,
             label=f"Observed dist_km (mean={obs_mean:.2f})")
    plt.hist(exp_dist, bins=bins, density=True, alpha=0.6,
             label=f"Predicted E[distance] (mean={exp_mean:.2f})")
    plt.xlabel("Distance (km)")
    plt.ylabel("Density")
    plt.title(f"Distance plausibility | {seg} | obs mean={obs_mean:.2f}, pred mean={exp_mean:.2f}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_attr_vs_access(Aj, L_in, S_pred, seg, out_png):
    plt.figure()
    s = 20 + 4000 * (S_pred / (S_pred.max() + 1e-12))
    plt.scatter(L_in, Aj, s=s, alpha=0.5)
    plt.xlabel("Incoming accessibility logsum (from origins, using EMU)")
    plt.ylabel("Attractivity index A_j = X_j beta (z-scale)")
    plt.title(f"Attractivity vs incoming accessibility | {seg}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_share_maps(tz_gdf, col, seg, out_png, title):
    plt.figure()
    ax = tz_gdf.plot(column=col, legend=True)
    ax.set_axis_off()
    plt.title(f"{title} | {seg}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()


# ============================================================
# MAIN
# ============================================================
set_all_seeds(BASE_SEED)

print("[LOAD] TZ features (withLIE)...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
if "npvm_id" not in tz.columns:
    raise ValueError("TZ layer missing 'npvm_id'")
tz["npvm_id"] = pd.to_numeric(tz["npvm_id"], errors="coerce")
tz = tz.dropna(subset=["npvm_id"]).copy()
tz["npvm_id"] = tz["npvm_id"].astype(int)

need_cols = ["npvm_id"] + BASE_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

print("[PREP] Global z-standardize destination variables on FULL TZ universe...")
tz_feat = tz.copy()
tz_feat, Z_COLS, MU_ALL, SD_ALL = standardize_all_zones(tz_feat, BASE_COLS, suffix="_z")

print("[LOAD] EMU matrix + mapping (withLIE)...")
emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

print("[LOAD] Average distance matrix (for distance plausibility)...")
dist_mat, zone_to_idx_dist, idx_to_zone_dist = load_matrix_and_mapping(DIST_OMX, DIST_MAT_NAME, MAP_NAME)
if len(idx_to_zone_dist) != len(idx_to_zone) or not np.all(idx_to_zone_dist == idx_to_zone):
    print("[WARN] Distance OMX mapping differs from utilities OMX mapping. Distance plausibility may be misaligned.")

rows_sampled = []
rows_full    = []
rows_coef    = []

seg_alpha = {}
seg_beta  = {}
seg_Aj_tz = {}
seg_Aj_by_omx = {}

# NEW
seg_vcov = {}
seg_se   = {}
seg_coef_vec = {}
seg_hsum = {}
seg_inference_meta = {}

tz_ids_master = np.array(sorted(tz_feat["npvm_id"].astype(int).unique().tolist()), dtype=int)

GPKG_OUT = os.path.join(OUT_DIR, "tz_outputs.gpkg")
if os.path.exists(GPKG_OUT):
    os.remove(GPKG_OUT)

for seg in SEGMENTS:
    print("\n" + "#" * 120)
    print(f"# SEGMENT = {seg} | K={K} | epochs={EPOCHS}")
    print("#" * 120)

    set_all_seeds(BASE_SEED + 1000 * SEG_SEED[seg])

    train_df = load_segment_csv(TRIPS_DIR, seg, "train")
    oos_df   = load_segment_csv(TRIPS_DIR, seg, "oos")

    seed_tr = BASE_SEED + 1000 * SEG_SEED[seg] + 10
    seed_te = BASE_SEED + 1000 * SEG_SEED[seg] + 20
    seed_full = FULL_EVAL_SEED + 1000 * SEG_SEED[seg]

    emu_tr, X_tr, y_tr, w_tr, train_df2, destset_tr, origidx_tr, destidx_tr = build_design(
        train_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_tr
    )
    emu_te, X_te, y_te, w_te, oos_df2, destset_te, origidx_te, destidx_te = build_design(
        oos_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_te
    )

    print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
    print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

    alpha_hat, beta_hat = train_mnl(
        emu_tr, X_tr, y_tr, w_tr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
    )

    seg_alpha[seg] = float(alpha_hat)
    seg_beta[seg]  = beta_hat.copy()

    coef_vec = np.concatenate([[float(alpha_hat)], beta_hat.astype(np.float64)])
    seg_coef_vec[seg] = coef_vec.copy()

    Xz_all_master = tz_feat.set_index("npvm_id").loc[tz_ids_master, Z_COLS].to_numpy(dtype=np.float32)
    Aj_master = Xz_all_master @ beta_hat.astype(np.float32)
    seg_Aj_tz[seg] = Aj_master.astype(np.float64)

    Msize = emu_mat.shape[0]
    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(tz_ids_master, Aj_master):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)
    seg_Aj_by_omx[seg] = Aj_by_omxidx

    met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)
    met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)

    rows_sampled.append({
        "segment": seg,
        "split": "train",
        "alpha_emu": float(alpha_hat),
        "K": K,
        **met_tr,
    })
    rows_sampled.append({
        "segment": seg,
        "split": "oos",
        "alpha_emu": float(alpha_hat),
        "K": K,
        **met_te,
    })

    print("[SAMPLED OOS ]",
          f"NLL_sum={met_te['NLL_sum']:.2f} NLL_avg={met_te['NLL_avg']:.4f} "
          f"MRR={met_te['MRR']:.4f} Top5={met_te['Top5']:.3f} Top10={met_te['Top10']:.3f} FF={met_te['FF']:.4f}")

    max_n = FULL_EVAL_MAX.get(seg, None)
    full_out, share_pack, dist_pack, trip_distance_df = full_eval_and_shares_oos_subset(
        oos_df=oos_df2,
        tz_feat=tz_feat,
        z_cols=Z_COLS,
        emu_mat=emu_mat,
        dist_mat=dist_mat,
        zone_to_idx=zone_to_idx,
        idx_to_zone=idx_to_zone,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=max_n,
        seed=seed_full,
        chunk=FULL_EVAL_CHUNK,
        topKs=TOPKS_FULL,
    )

    rows_full.append({
        "segment": seg,
        "alpha_emu": float(alpha_hat),
        "K": K,
        **full_out,
    })

    print("[FULL OOS   ]",
          f"NLL_full_avg={full_out['NLL_full_avg']:.4f} MRR_full={full_out['MRR_full']:.4f} "
          f"FF_full={full_out['FF_full']:.6f} "
          f"R2_full={full_out['McFadden_R2_full']:.4f} Top5_full={full_out['Top5_full']:.3f} Top10_full={full_out['Top10_full']:.3f}")
    print("[SHARES FULL]",
          f"Pear={full_out['Pearson_share']:.4f} Spear={full_out['Spearman_share']:.4f} "
          f"MAE={full_out['MAE_share']:.6f} RMSE={full_out['RMSE_share']:.6f} JS={full_out['JS_share']:.6f}")

    obs_dist, exp_dist = dist_pack
    if (obs_dist is not None) and (exp_dist is not None):
        png = os.path.join(PLOT_DIR, f"distance_plausibility_{seg}.png")
        plot_distance_plausibility(obs_dist, exp_dist, seg, png)
        print(f"[PLOT] wrote {png}")

        if trip_distance_df is not None:
            dist_csv = os.path.join(DIST_OOS_DIR, f"distance_triplevel_{seg}.csv")
            trip_distance_df.to_csv(dist_csv, index=False)
            print(f"[CSV ] wrote {dist_csv}")

    # =======================================================
    # Hessian-based full inference objects
    # =======================================================
    vcov_mat, se_vec, H_sum, used_n, used_sum_w = approx_hessian_inference(
        emu_set=emu_tr,
        X_set=X_tr,
        y=y_tr,
        w=w_tr,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=SE_MAX_TRAIN.get(seg, 20000),
        seed=SE_SEED + 1000 * SEG_SEED[seg],
        ridge=1e-8,
    )

    seg_vcov[seg] = vcov_mat.copy()
    seg_se[seg]   = se_vec.copy()
    seg_hsum[seg] = H_sum.copy()
    seg_inference_meta[seg] = {
        "segment": seg,
        "used_n_for_hessian": used_n,
        "used_sum_w_for_hessian": used_sum_w,
        "num_params": len(PARAM_NAMES),
    }

    # export vcov and hessian
    df_vcov = pd.DataFrame(vcov_mat, index=PARAM_NAMES, columns=PARAM_NAMES)
    df_hsum = pd.DataFrame(H_sum, index=PARAM_NAMES, columns=PARAM_NAMES)
    df_vcov.to_csv(os.path.join(VCOV_DIR, f"vcov_{seg}.csv"))
    df_hsum.to_csv(os.path.join(VCOV_DIR, f"hessian_sum_{seg}.csv"))

    # export per-segment diagonal summary
    df_diag = pd.DataFrame({
        "segment": seg,
        "param": PARAM_NAMES,
        "coef": coef_vec,
        "se": se_vec,
        "var": np.diag(vcov_mat),
    })
    df_diag.to_csv(os.path.join(VCOV_DIR, f"coef_se_diag_{seg}.csv"), index=False)

    # coefficient table rows
    for name, b, s in zip(PARAM_NAMES, coef_vec.tolist(), se_vec.tolist()):
        zval = b / (s + 1e-12)
        pval = p_value_two_sided_z(zval)
        rows_coef.append({
            "segment": seg,
            "param": name,
            "coef": float(b),
            "se": float(s),
            "z": float(zval),
            "p": float(pval),
            "stars": stars(pval),
        })

    if share_pack is not None:
        tz_ids  = share_pack["tz_ids"]
        S_obs   = share_pack["S_obs"]
        S_pred  = share_pack["S_pred"]
        Aj      = share_pack["Aj"]
        orig_i  = share_pack["orig_idx"]
        w_i     = share_pack["w"]

        tz_pos = {int(z): i for i, z in enumerate(tz_ids.tolist())}

        L_in = incoming_accessibility_logsum(
            tz_ids=tz_ids,
            tz_pos=tz_pos,
            emu_mat=emu_mat,
            alpha=float(alpha_hat),
            orig_idx=orig_i,
            w=w_i,
            zone_to_idx=zone_to_idx,
            idx_to_zone=idx_to_zone,
            chunk=FULL_EVAL_CHUNK,
        )

        png = os.path.join(PLOT_DIR, f"attr_vs_access_{seg}.png")
        plot_attr_vs_access(Aj, L_in, S_pred, seg, png)
        print(f"[PLOT] wrote {png}")

        tz_out = tz_feat[["npvm_id", "geometry"]].copy()
        tz_out = tz_out.set_index("npvm_id").loc[tz_ids].reset_index()

        tz_out[f"A_attr_{seg}"] = Aj
        tz_out[f"S_obs_{seg}"]  = S_obs
        tz_out[f"S_pred_{seg}"] = S_pred
        tz_out[f"L_in_{seg}"]   = L_in

        png1 = os.path.join(PLOT_DIR, f"map_S_pred_{seg}.png")
        png2 = os.path.join(PLOT_DIR, f"map_A_attr_{seg}.png")
        plot_share_maps(tz_out, f"S_pred_{seg}", seg, png1, "Predicted destination share (OOS subset, full denom)")
        plot_share_maps(tz_out, f"A_attr_{seg}", seg, png2, "Attractivity index A_j = X_j beta (z-scale)")
        print(f"[PLOT] wrote {png1}")
        print(f"[PLOT] wrote {png2}")

        layer_name = f"TZ_{seg}"
        tz_out.to_file(GPKG_OUT, layer=layer_name, driver="GPKG")
        print(f"[GPKG] wrote layer {layer_name} -> {GPKG_OUT}")


# ============================================================
# Save summary tables
# ============================================================
df_sampled = pd.DataFrame(rows_sampled)
df_full    = pd.DataFrame(rows_full)
df_coef    = pd.DataFrame(rows_coef)
df_infmeta = pd.DataFrame(list(seg_inference_meta.values()))

p1 = os.path.join(OUT_DIR, "metrics_sampled.csv")
p2 = os.path.join(OUT_DIR, "metrics_full_and_shares.csv")
p3 = os.path.join(OUT_DIR, "coef_table.csv")
p4 = os.path.join(OUT_DIR, "hessian_inference_meta.csv")

df_sampled.to_csv(p1, index=False)
df_full.to_csv(p2, index=False)
df_coef.to_csv(p3, index=False)
df_infmeta.to_csv(p4, index=False)

# ============================================================
# NEW: coefficient difference tests
# ============================================================
rows_within = []
for seg in SEGMENTS:
    rows_within.extend(
        build_within_segment_diff_tests(
            segment=seg,
            params=PARAM_NAMES,
            coef_vec=seg_coef_vec[seg],
            vcov=seg_vcov[seg],
        )
    )
df_within = pd.DataFrame(rows_within)
df_within.to_csv(os.path.join(TEST_DIR, "coef_diff_within_segment.csv"), index=False)

rows_across = build_across_segment_sameparam_tests(
    seg_to_coef=seg_coef_vec,
    seg_to_vcov=seg_vcov,
    params=PARAM_NAMES,
)
df_across = pd.DataFrame(rows_across)
df_across.to_csv(os.path.join(TEST_DIR, "coef_diff_across_segments.csv"), index=False)

# helpful wide summary for same parameter across all segments
wide_coef = df_coef.pivot(index="param", columns="segment", values="coef").reset_index()
wide_se   = df_coef.pivot(index="param", columns="segment", values="se").reset_index()
wide_coef.to_csv(os.path.join(TEST_DIR, "coef_wide_by_segment.csv"), index=False)
wide_se.to_csv(os.path.join(TEST_DIR, "se_wide_by_segment.csv"), index=False)

print("\n" + "=" * 110)
print("[OK] wrote", p1)
print("[OK] wrote", p2)
print("[OK] wrote", p3)
print("[OK] wrote", p4)
print("[OK] wrote", GPKG_OUT)
print("[OK] vcov matrices in", VCOV_DIR)
print("[OK] coefficient tests in", TEST_DIR)
print("[OK] plots in", PLOT_DIR)
print("[OK] trip-level distance CSVs in", DIST_OOS_DIR)
print("=" * 110)

print("\nSAMPLED OOS METRICS (weighted):")
print(df_sampled[df_sampled["split"] == "oos"][["segment","NLL_sum","NLL_avg","MRR","Top5","Top10","FF","N","J","sum_WP"]]
      .sort_values("segment").to_string(index=False))

print("\nFULL OOS METRICS + SHARE DIAGNOSTICS:")
keep_cols = [
    "segment","NLL_full_sum","NLL_full_avg","MRR_full","FF_full","McFadden_R2_full","Top5_full","Top10_full",
    "Pearson_share","Spearman_share","MAE_share","RMSE_share","JS_share","N_full_eval","WP_full_eval","J_full",
    "dist_obs_mean_km","dist_pred_mean_km","dist_mean_diff_km","dist_obs_p50_km","dist_pred_p50_km","dist_obs_p90_km","dist_pred_p90_km",
]
cols_exist = [c for c in keep_cols if c in df_full.columns]
print(df_full[cols_exist].sort_values("segment").to_string(index=False))

print("\nTOP OF coef_diff_across_segments.csv")
print(df_across.head(20).to_string(index=False))

# ============================================================
# EXPORT "Result" PACKAGE FOR SIMBA INTEGRATION
# ============================================================
print("\n" + "=" * 110)
print("[EXPORT] Building Result package (Attractivity + utilities omx) ...")

omx_idx_master = np.array([zone_to_idx.get(int(z), -1) for z in tz_ids_master], dtype=int)

attr_df = pd.DataFrame({
    "zone_id": tz_ids_master.astype(int),
    "omx_idx": omx_idx_master.astype(int),
    "Attr_YS": seg_Aj_tz["YS"].astype(np.float64),
    "Attr_OS": seg_Aj_tz["OS"].astype(np.float64),
    "Attr_YL": seg_Aj_tz["YL"].astype(np.float64),
    "Attr_OL": seg_Aj_tz["OL"].astype(np.float64),
})

attr_df.to_csv(ATTR_CSV_OUT, index=False)
print("[OK] wrote", ATTR_CSV_OUT)

if os.path.exists(ATTR_GPKG_OUT):
    os.remove(ATTR_GPKG_OUT)

geom = tz_feat[["npvm_id", "geometry"]].drop_duplicates().set_index("npvm_id").loc[tz_ids_master].reset_index()
geom = geom.rename(columns={"npvm_id": "zone_id"})
attr_gdf = gpd.GeoDataFrame(
    geom.merge(attr_df, on="zone_id", how="left"),
    geometry="geometry",
    crs=tz_feat.crs,
)
attr_gdf.to_file(ATTR_GPKG_OUT, driver="GPKG")
print("[OK] wrote", ATTR_GPKG_OUT)

if os.path.exists(UTIL_SEG_OMX):
    os.remove(UTIL_SEG_OMX)

Msize = emu_mat.shape[0]
print(f"[OMX-OUT] Creating {UTIL_SEG_OMX} with 4 matrices of shape {emu_mat.shape} (float32) ...")

fout = omx.open_file(UTIL_SEG_OMX, "w")
try:
    fout.create_mapping(MAP_NAME, idx_to_zone.astype(np.int32))
except Exception:
    fout.create_mapping(MAP_NAME, [int(x) for x in idx_to_zone.tolist()])

for seg in SEGMENTS:
    float_atom = tb.Atom.from_dtype(np.dtype("float32"))
    fout.create_matrix(f"utility_{seg}", atom=float_atom, shape=(Msize, Msize))

ROW_BLOCK = 512
for seg in SEGMENTS:
    alpha32 = np.float32(seg_alpha[seg])
    Aj_omx = seg_Aj_by_omx[seg].astype(np.float32)

    print(f"[OMX-OUT] Writing utility_{seg} ... alpha={float(seg_alpha[seg]):.6f}")
    mat = fout[f"utility_{seg}"]

    for i0 in range(0, Msize, ROW_BLOCK):
        i1 = min(Msize, i0 + ROW_BLOCK)
        emu_blk = emu_mat[i0:i1, :]
        util_blk = alpha32 * emu_blk + Aj_omx[None, :]
        mat[i0:i1, :] = util_blk.astype(np.float32)

fout.close()
print("[OK] wrote", UTIL_SEG_OMX)

print("[EXPORT] Done. Result files are in:", RESULT_DIR)
print("=" * 110)

## Proposed Specification: coefficient anlaysis post estimation

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd

# ============================================================
# COEFFICIENT DIFFERENCE ANALYSIS
# ------------------------------------------------------------
# This script reads precomputed coefficient-difference tables
# and creates clean summaries for:
#
#   1) Within-segment coefficient comparisons
#      -> Is beta_a significantly different from beta_b
#         inside the same segment?
#
#   2) Across-segment same-coefficient comparisons
#      -> Is beta_h significantly different between segments?
#
# INPUTS (already available from your previous pipeline):
#   - coef_diff_across_segments.csv
#   - coef_diff_within_segment.csv
#   - coef_wide_by_segment.csv
#   - se_wide_by_segment.csv
#
# OUTPUTS:
#   - summary tables with significant differences
#   - readable yes/no tables
#   - top contrasts
# ============================================================

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
BASE_DIR = "ModelRuns/BaselineFinal2/coef_tests"
OUT_DIR = os.path.join(BASE_DIR, "summaries_clean")
os.makedirs(OUT_DIR, exist_ok=True)

ACROSS_FILE = os.path.join(BASE_DIR, "coef_diff_across_segments.csv")
WITHIN_FILE = os.path.join(BASE_DIR, "coef_diff_within_segment.csv")
COEF_WIDE_FILE = os.path.join(BASE_DIR, "coef_wide_by_segment.csv")
SE_WIDE_FILE = os.path.join(BASE_DIR, "se_wide_by_segment.csv")

# ------------------------------------------------------------
# Parameters to keep
# - INCLUDE_ALPHA = True also includes alpha_emu
# ------------------------------------------------------------
INCLUDE_ALPHA = True

FINAL_ORDER = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F6_others_count_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
    "F10_superinfra_log1p",
]

if INCLUDE_ALPHA:
    FINAL_ORDER_WITH_ALPHA = FINAL_ORDER + ["alpha_emu"]
else:
    FINAL_ORDER_WITH_ALPHA = FINAL_ORDER.copy()

SEGMENT_ORDER = ["YS", "OS", "YL", "OL"]
SEGMENT_PAIRS = list(itertools.combinations(SEGMENT_ORDER, 2))

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def significance_label(p):
    """Return significance stars from p-value."""
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.10:
        return "."
    return ""

def sig_yes_no(p, alpha=0.05):
    """Readable yes/no flag for significance."""
    if pd.isna(p):
        return ""
    return "yes" if p < alpha else "no"

def clean_param_name(x):
    """Optional pretty label mapping for output tables."""
    mapping = {
        "F1_gastr_count_log1p": "F1_gastr_count",
        "F2_pop_total_log1p": "F2_pop_total",
        "F8_outdoor_lake_raw_dens_log1p": "F8_outdoor_lake_raw_dens",
        "F3_outdoor_hard_count_log1p": "F3_outdoor_hard_count",
        "F3_outdoor_soft_count_log1p": "F3_outdoor_soft_count",
        "F3_outdoor_LUmix": "F3_outdoor_LUmix",
        "F4_cult_count_log1p": "F4_cult_count",
        "F5_sport_count_log1p": "F5_sport_count",
        "F5_sport_out_length_log1p": "F5_sport_out_length",
        "F6_others_count_log1p": "F6_others_count",
        "F7_urban_sum_log1p": "F7_urban_sum",
        "F8_POI_urban_dens_log1p": "F8_POI_urban_dens",
        "F10_superinfra_log1p": "F10_superinfra",
        "alpha_emu": "alpha_EMU",
    }
    return mapping.get(x, x)

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
for path in [ACROSS_FILE, WITHIN_FILE, COEF_WIDE_FILE, SE_WIDE_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

df_across = pd.read_csv(ACROSS_FILE)
df_within = pd.read_csv(WITHIN_FILE)
df_coef_wide = pd.read_csv(COEF_WIDE_FILE)
df_se_wide = pd.read_csv(SE_WIDE_FILE)

print("[OK] Files loaded.")
print("Across shape :", df_across.shape)
print("Within shape :", df_within.shape)
print("Coef wide    :", df_coef_wide.shape)
print("SE wide      :", df_se_wide.shape)

# ------------------------------------------------------------
# Keep only selected parameters
# ------------------------------------------------------------
df_across = df_across[df_across["param"].isin(FINAL_ORDER_WITH_ALPHA)].copy()
df_within = df_within[
    df_within["param_1"].isin(FINAL_ORDER_WITH_ALPHA)
    & df_within["param_2"].isin(FINAL_ORDER_WITH_ALPHA)
].copy()

# Remove alpha_emu from within-segment comparisons if desired
if not INCLUDE_ALPHA:
    df_within = df_within[
        (df_within["param_1"] != "alpha_emu") &
        (df_within["param_2"] != "alpha_emu")
    ].copy()

# ------------------------------------------------------------
# Add pretty names
# ------------------------------------------------------------
df_across["param_clean"] = df_across["param"].map(clean_param_name)
df_within["param_1_clean"] = df_within["param_1"].map(clean_param_name)
df_within["param_2_clean"] = df_within["param_2"].map(clean_param_name)

# ============================================================
# PART 1 — ACROSS-SEGMENT TESTS
# ============================================================

across_sig_005 = df_across[df_across["p_delta"] < 0.05].copy()
across_sig_001 = df_across[df_across["p_delta"] < 0.01].copy()

across_sig_005 = across_sig_005.sort_values(["param", "p_delta", "segment_1", "segment_2"])
across_sig_001 = across_sig_001.sort_values(["param", "p_delta", "segment_1", "segment_2"])

across_sig_005.to_csv(os.path.join(OUT_DIR, "across_segments_significant_p005.csv"), index=False)
across_sig_001.to_csv(os.path.join(OUT_DIR, "across_segments_significant_p001.csv"), index=False)

# ------------------------------------------------------------
# Compact readable across-segment summary
# ------------------------------------------------------------
summary_rows = []

for param in FINAL_ORDER_WITH_ALPHA:
    sub = df_across[df_across["param"] == param].copy()
    row = {
        "param": param,
        "param_clean": clean_param_name(param),
        "n_significant_pairs_p005": int((sub["p_delta"] < 0.05).sum()),
        "n_significant_pairs_p001": int((sub["p_delta"] < 0.01).sum()),
    }

    for s1, s2 in SEGMENT_PAIRS:
        tmp = sub[(sub["segment_1"] == s1) & (sub["segment_2"] == s2)]
        if len(tmp) == 1:
            tmp = tmp.iloc[0]
            row[f"{s1}_vs_{s2}_delta"] = tmp["delta_1_minus_2"]
            row[f"{s1}_vs_{s2}_p"] = tmp["p_delta"]
            row[f"{s1}_vs_{s2}_sig"] = sig_yes_no(tmp["p_delta"], alpha=0.05)
            row[f"{s1}_vs_{s2}_stars"] = significance_label(tmp["p_delta"])
        else:
            row[f"{s1}_vs_{s2}_delta"] = np.nan
            row[f"{s1}_vs_{s2}_p"] = np.nan
            row[f"{s1}_vs_{s2}_sig"] = ""
            row[f"{s1}_vs_{s2}_stars"] = ""
    summary_rows.append(row)

summary_across = pd.DataFrame(summary_rows)
summary_across.to_csv(os.path.join(OUT_DIR, "summary_across_segments_by_parameter.csv"), index=False)

# ------------------------------------------------------------
# Pretty yes/no across-segment matrix
# ------------------------------------------------------------
pretty_rows = []

for param in FINAL_ORDER_WITH_ALPHA:
    sub = df_across[df_across["param"] == param].copy()
    row = {
        "param": param,
        "param_clean": clean_param_name(param),
    }

    for s1, s2 in SEGMENT_PAIRS:
        tmp = sub[(sub["segment_1"] == s1) & (sub["segment_2"] == s2)]
        if len(tmp) == 1:
            tmp = tmp.iloc[0]
            delta = float(tmp["delta_1_minus_2"])
            pval = float(tmp["p_delta"])
            direction = "+" if delta > 0 else "-"
            row[f"{s1}_vs_{s2}"] = f"{direction} sig" if pval < 0.05 else f"{direction} ns"
        else:
            row[f"{s1}_vs_{s2}"] = ""
    pretty_rows.append(row)

pretty_across = pd.DataFrame(pretty_rows)
pretty_across.to_csv(os.path.join(OUT_DIR, "pretty_across_segments_yesno.csv"), index=False)

# ============================================================
# PART 2 — WITHIN-SEGMENT TESTS
# ============================================================

within_sig_005 = df_within[df_within["p_delta"] < 0.05].copy()
within_sig_001 = df_within[df_within["p_delta"] < 0.01].copy()

within_sig_005 = within_sig_005.sort_values(["segment", "p_delta", "param_1", "param_2"])
within_sig_001 = within_sig_001.sort_values(["segment", "p_delta", "param_1", "param_2"])

within_sig_005.to_csv(os.path.join(OUT_DIR, "within_segment_significant_p005.csv"), index=False)
within_sig_001.to_csv(os.path.join(OUT_DIR, "within_segment_significant_p001.csv"), index=False)

# ------------------------------------------------------------
# Top coefficient contrasts within each segment
# ------------------------------------------------------------
top_within_rows = []
for seg in SEGMENT_ORDER:
    sub = df_within[df_within["segment"] == seg].copy()
    sub["abs_delta"] = sub["delta_1_minus_2"].abs()
    sub = sub.sort_values("abs_delta", ascending=False)
    top_within_rows.append(sub.head(20))

top_within = pd.concat(top_within_rows, ignore_index=True)
top_within.to_csv(os.path.join(OUT_DIR, "top_within_segment_contrasts.csv"), index=False)

# ------------------------------------------------------------
# Within-segment counts
# ------------------------------------------------------------
within_count_rows = []
for seg in SEGMENT_ORDER:
    sub = df_within[df_within["segment"] == seg].copy()
    within_count_rows.append({
        "segment": seg,
        "n_total_pairs": len(sub),
        "n_significant_pairs_p005": int((sub["p_delta"] < 0.05).sum()),
        "n_significant_pairs_p001": int((sub["p_delta"] < 0.01).sum()),
    })

within_counts = pd.DataFrame(within_count_rows)
within_counts.to_csv(os.path.join(OUT_DIR, "within_segment_significant_counts.csv"), index=False)

# ============================================================
# PART 3 — PARAMETER-LEVEL FOCUS TABLES
# ============================================================
param_dir = os.path.join(OUT_DIR, "by_parameter")
os.makedirs(param_dir, exist_ok=True)

for param in FINAL_ORDER_WITH_ALPHA:
    sub = df_across[df_across["param"] == param].copy()
    sub = sub.sort_values(["segment_1", "segment_2"])
    out_path = os.path.join(param_dir, f"across_{param}.csv")
    sub.to_csv(out_path, index=False)

seg_dir = os.path.join(OUT_DIR, "by_segment")
os.makedirs(seg_dir, exist_ok=True)

for seg in SEGMENT_ORDER:
    sub = df_within[df_within["segment"] == seg].copy()
    sub = sub.sort_values(["p_delta", "param_1", "param_2"])
    out_path = os.path.join(seg_dir, f"within_{seg}.csv")
    sub.to_csv(out_path, index=False)

# ============================================================
# PART 4 — SCREEN OUTPUT
# ============================================================
print("\n" + "=" * 90)
print("TOP ACROSS-SEGMENT DIFFERENCES (p < 0.05)")
print("=" * 90)
print(
    across_sig_005[
        ["param_clean", "segment_1", "segment_2", "coef_1", "coef_2",
         "delta_1_minus_2", "se_delta", "z_delta", "p_delta", "stars_delta"]
    ]
    .head(40)
    .to_string(index=False)
)

print("\n" + "=" * 90)
print("TOP WITHIN-SEGMENT DIFFERENCES (p < 0.05)")
print("=" * 90)
print(
    within_sig_005[
        ["segment", "param_1_clean", "param_2_clean", "coef_1", "coef_2",
         "delta_1_minus_2", "se_delta", "z_delta", "p_delta", "stars_delta"]
    ]
    .head(40)
    .to_string(index=False)
)

print("\n" + "=" * 90)
print("WITHIN-SEGMENT SIGNIFICANT COUNTS")
print("=" * 90)
print(within_counts.to_string(index=False))

print("\n[OK] All outputs saved in:")
print(OUT_DIR)

## Accessibility (EMU) only specification

In [ ]:
# ============================================================
# BASELINE EMU-ONLY (withLIE, byPerson) -- NO destination variables
#
# Aligned 1:1 with your updated baseline script:
#   - sampled training on chosen + K non-chosen alternatives
#   - sampled OOS metrics: NLL, Top5/10, MRR, FF
#   - FULL-choice-set OOS metrics: NLL_full, MRR_full, FF_full,
#     McFadden_R2_full, Top5/10_full
#   - aggregate destination-share diagnostics:
#     Pearson, Spearman, MAE, RMSE, JS
#   - distance plausibility using avg_distances_ready
#   - SAME bins for observed vs predicted distance histograms
#   - means shown in title/legend
#   - trip-level OOS distance CSV export:
#       orig_zone, dest_zone_obs, WP, dist_obs_km, dist_exp_km
#   - coefficient inference via Hessian-based SE (alpha only)
#   - map-ready TZ outputs (S_obs / S_pred)
#
# Outputs under OUT_DIR:
#   - metrics_sampled.csv
#   - metrics_full_and_shares.csv
#   - coef_table.csv
#   - tz_outputs.gpkg
#   - plots/*.png
#   - distance_oos_triplevel/distance_triplevel_<SEG>.csv
# ============================================================

import os
import math
import random
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import openmatrix as omx
from pathlib import Path
import matplotlib.pyplot as plt


# -----------------------------
# CONFIG
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"
DIST_COL  = "dist_km"

TRIPS_DIR = "Trips/byPerson"  # WITH Liechtenstein
TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")

# Impedance
UTIL_OMX      = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME  = "utility_emu"

# Distance plausibility: average network distance
DIST_OMX      = READY_DIR / "distance_avg_2023_ready.omx"
DIST_MAT_NAME = "avg_distances_ready"

MAP_NAME = "NO"

OUT_DIR  = "ModelRuns/BaselineEMUOnly"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DIST_OOS_DIR = os.path.join(OUT_DIR, "distance_oos_triplevel")
os.makedirs(DIST_OOS_DIR, exist_ok=True)

# sampled training
K = 1000
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# full eval
FULL_EVAL_MAX = {"YS": 20000, "OS": 8000, "YL": 8000, "OL": 8000}
FULL_EVAL_SEED  = 777
FULL_EVAL_CHUNK = 256

# Hessian SEs on train subset
SE_MAX_TRAIN = {"YS": 40000, "OS": 25000, "YL": 12000, "OL": 6000}
SE_SEED = 2024

BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

TOPKS_SAMPLED = (5, 10)
TOPKS_FULL    = (5, 10)


# ============================================================
# Reproducibility
# ============================================================
def set_all_seeds(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Small helpers
# ============================================================
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def p_value_two_sided_z(z):
    return 2.0 * (1.0 - norm_cdf(abs(float(z))))

def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.1:
        return "."
    return ""


# ============================================================
# IO helpers
# ============================================================
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()

    if DIST_COL in df.columns:
        df[DIST_COL] = pd.to_numeric(df[DIST_COL], errors="coerce")

    return df

def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone


# ============================================================
# Sampling + EMU-only design
# ============================================================
def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray, k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0] = c
        dest_set[i, 1:] = alts
    return dest_set

def build_design_emu_only(df: pd.DataFrame, tz_ids: np.ndarray, emu_mat: np.ndarray,
                          zone_to_idx: dict, k: int, seed: int):
    """
    Returns:
      emu_set: (N,K+1) tensor
      y: (N,) zeros
      w: (N,) tensor
      df2, dest_set2 (zone ids), orig_idx2, dest_idx2 (OMX indices)
    """
    rng = np.random.default_rng(seed)
    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    dest_set = sample_choice_sets_unique(chosen, tz_ids, k=k, rng=rng)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    flat = dest_set.ravel()
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(len(df), k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2  = w[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]
    dest_set2 = dest_set[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
        dest_set2,
        orig_idx2,
        dest_idx2,
    )


# ============================================================
# EMU-only model: V = alpha * EMU
# ============================================================
def train_mnl_emu_only(emu_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_alpha = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            e_b = emu_set[b]
            y_b = y[b]
            w_b = w[b]

            V = alpha * e_b
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum()

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted-avg NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_alpha = float(alpha.detach().cpu())

    return best_alpha

def eval_sampled_metrics_emu_only(emu_set, w, alpha, topKs=(5, 10), want_ff=True):
    with torch.no_grad():
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)
        V = alpha_t * emu_set
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        P = torch.exp(logP)

        chosen_logp = logP[:, 0]
        chosen_p    = P[:, 0]

        sum_w = float(w.sum().cpu())
        nll_sum = float((-(w * chosen_logp)).sum().cpu())
        nll_avg = nll_sum / (sum_w + 1e-9)

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()
        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))

        out = {
            "NLL_sum": nll_sum,
            "NLL_avg": nll_avg,
            "MRR": mrr,
            "sum_WP": sum_w,
            "N": int(V.shape[0]),
            "J": int(V.shape[1]),
        }
        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))
        if want_ff:
            out["FF"] = float((w * chosen_p).sum().cpu() / (sum_w + 1e-9))
        return out


# ============================================================
# FULL eval + shares (EMU-only)
# ============================================================
def full_eval_and_shares_oos_subset_emu_only(
    oos_df: pd.DataFrame,
    tz_ids: np.ndarray,
    emu_mat: np.ndarray,
    dist_mat: np.ndarray | None,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(5, 10),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w    = df[WP_COL].to_numpy(dtype=np.float64)

    has_dist = (DIST_COL in df.columns) and df[DIST_COL].notna().any()
    obs_dist = df[DIST_COL].to_numpy(dtype=np.float64) if has_dist else None

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)
    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)

    if has_dist:
        good = good & np.isfinite(obs_dist) & (obs_dist >= 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w        = w[good]
    if has_dist:
        obs_dist = obs_dist[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}, None, (None, None), None

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    tz_set = set(int(z) for z in tz_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(tz_ids.tolist())}

    pred_mass = np.zeros(len(tz_ids), dtype=np.float64)
    obs_mass  = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(tz_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    ll_model = 0.0
    mrr_num = 0.0
    ff_num = 0.0
    top_num = {k: 0.0 for k in topKs}
    ranks   = np.empty(len(df), dtype=np.int64)

    exp_dist = np.zeros(len(df), dtype=np.float64) if (dist_mat is not None and has_dist) else None

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d])

        # pass 1: max utility
        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32)
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        # pass 2: sumexp + rank
        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32)
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse
        p_chosen = math.exp(logp_chosen)

        nll_num  += wn * (-logp_chosen)
        ll_model += wn * logp_chosen
        ff_num   += wn * p_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        if exp_dist is not None:
            expd = 0.0

        # pass 3: shares + expected distance
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32)
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

            if exp_dist is not None:
                dist_blk = dist_mat[o, j0:j1].astype(np.float64)
                expd += float(np.sum(Pblk * dist_blk))

        if exp_dist is not None:
            exp_dist[n] = expd

    nll_full = float(nll_num)
    nll_full_avg = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    ff_full = float(ff_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs  = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)

    pear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="pearson")) if len(S_obs) > 2 else np.nan
    spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    mae  = float(np.mean(np.abs(S_obs - S_pred)))
    rmse = float(np.sqrt(np.mean((S_obs - S_pred) ** 2)))
    js   = float(js_div(S_obs, S_pred))

    dist_summary = {}
    if exp_dist is not None:
        wnorm = w / (sum_w + 1e-12)
        obs_mean = float(np.sum(wnorm * obs_dist))
        pred_mean = float(np.sum(wnorm * exp_dist))
        dist_summary = {
            "dist_obs_mean_km": obs_mean,
            "dist_pred_mean_km": pred_mean,
            "dist_mean_diff_km": float(pred_mean - obs_mean),
            "dist_obs_p50_km": float(np.quantile(obs_dist, 0.50)),
            "dist_pred_p50_km": float(np.quantile(exp_dist, 0.50)),
            "dist_obs_p90_km": float(np.quantile(obs_dist, 0.90)),
            "dist_pred_p90_km": float(np.quantile(exp_dist, 0.90)),
        }

    out = {
        "NLL_full_sum": nll_full,
        "NLL_full_avg": nll_full_avg,
        "MRR_full": mrr_full,
        "FF_full": ff_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "Pearson_share": pear,
        "Spearman_share": spear,
        "MAE_share": mae,
        "RMSE_share": rmse,
        "JS_share": js,
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    out.update(dist_summary)

    trip_distance_df = None
    if exp_dist is not None:
        trip_distance_df = pd.DataFrame({
            "orig_zone": df[ORIG_COL].to_numpy(dtype=int),
            "dest_zone_obs": df[DEST_COL].to_numpy(dtype=int),
            "WP": w.astype(np.float64),
            "dist_obs_km": obs_dist.astype(np.float64),
            "dist_exp_km": exp_dist.astype(np.float64),
        })

    return out, {
        "tz_ids": tz_ids,
        "S_obs": S_obs,
        "S_pred": S_pred,
        "orig_idx": orig_idx,
        "w": w,
    }, (obs_dist, exp_dist), trip_distance_df


# ============================================================
# Hessian-based SE (alpha only)
# ============================================================
def approx_hessian_se_alpha_only(
    emu_set: torch.Tensor,
    y: torch.Tensor,
    w: torch.Tensor,
    alpha_hat: float,
    max_n: int,
    seed: int,
    ridge: float = 1e-8,
):
    """
    Hessian-based SE aligned with your baseline logic, but alpha-only.
    """
    N = emu_set.shape[0]
    rng = np.random.default_rng(seed)

    if N > max_n:
        idx = rng.choice(N, size=max_n, replace=False)
        idx = torch.tensor(idx, dtype=torch.long, device=DEVICE)
        emu = emu_set[idx]
        yy  = y[idx]
        ww  = w[idx]
    else:
        emu, yy, ww = emu_set, y, w

    sum_w = float(ww.sum().detach().cpu())

    alpha = torch.tensor([alpha_hat], dtype=DTYPE, device=DEVICE, requires_grad=True)

    def loss_avg(a):
        V = a * emu
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        ll = logP[torch.arange(V.shape[0], device=DEVICE), yy]
        return (-(ww * ll)).sum() / (ww.sum() + 1e-12)

    L = loss_avg(alpha)
    g = torch.autograd.grad(L, [alpha], create_graph=True)[0]
    H_avg = torch.autograd.grad(g, [alpha])[0].detach().cpu().numpy().reshape(1, 1)

    H_sum = sum_w * H_avg
    H_sum = 0.5 * (H_sum + H_sum.T)
    H_sum = H_sum + ridge * np.eye(1, dtype=np.float64)

    Vcov = np.linalg.pinv(H_sum)
    se = float(np.sqrt(np.clip(Vcov[0, 0], 0.0, np.inf)))
    return se


# ============================================================
# Plots
# ============================================================
def plot_distance_plausibility(obs_dist, exp_dist, seg, out_png):
    obs_dist = np.asarray(obs_dist, dtype=np.float64)
    exp_dist = np.asarray(exp_dist, dtype=np.float64)

    finite = np.isfinite(obs_dist) & np.isfinite(exp_dist)
    obs_dist = obs_dist[finite]
    exp_dist = exp_dist[finite]

    if len(obs_dist) == 0:
        return

    obs_mean = float(np.mean(obs_dist))
    exp_mean = float(np.mean(exp_dist))

    xmax = float(max(np.max(obs_dist), np.max(exp_dist)))
    bins = np.linspace(0.0, xmax, 51)  # same 50 bins

    plt.figure()
    plt.hist(obs_dist, bins=bins, density=True, alpha=0.6,
             label=f"Observed dist_km (mean={obs_mean:.2f})")
    plt.hist(exp_dist, bins=bins, density=True, alpha=0.6,
             label=f"Predicted E[distance] (mean={exp_mean:.2f})")
    plt.xlabel("Distance (km)")
    plt.ylabel("Density")
    plt.title(f"Distance plausibility | {seg} | EMU-only | obs mean={obs_mean:.2f}, pred mean={exp_mean:.2f}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_share_map(tz_gdf, col, seg, out_png):
    plt.figure()
    ax = tz_gdf.plot(column=col, legend=True)
    ax.set_axis_off()
    plt.title(f"S_pred (EMU-only) | {seg}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()


# ============================================================
# MAIN
# ============================================================
set_all_seeds(BASE_SEED)

print("[LOAD] TZ universe (ids + geometry)...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
if "npvm_id" not in tz.columns:
    raise ValueError("TZ layer missing 'npvm_id'")
tz["npvm_id"] = pd.to_numeric(tz["npvm_id"], errors="coerce")
tz = tz.dropna(subset=["npvm_id"]).copy()
tz["npvm_id"] = tz["npvm_id"].astype(int)
tz_ids = np.array(sorted(tz["npvm_id"].unique().tolist()), dtype=int)

print("[LOAD] EMU matrix + mapping...")
emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

print("[LOAD] Average distance matrix (for distance plausibility)...")
dist_mat, zone_to_idx_dist, idx_to_zone_dist = load_matrix_and_mapping(DIST_OMX, DIST_MAT_NAME, MAP_NAME)
if len(idx_to_zone_dist) != len(idx_to_zone) or not np.all(idx_to_zone_dist == idx_to_zone):
    print("[WARN] Distance mapping differs from utilities mapping. Distance plausibility may be misaligned.")

rows_sampled = []
rows_full    = []
rows_coef    = []

GPKG_OUT = os.path.join(OUT_DIR, "tz_outputs.gpkg")
if os.path.exists(GPKG_OUT):
    os.remove(GPKG_OUT)

for seg in SEGMENTS:
    print("\n" + "#" * 120)
    print(f"# SEGMENT = {seg} | EMU-only | K={K} | epochs={EPOCHS}")
    print("#" * 120)

    set_all_seeds(BASE_SEED + 1000 * SEG_SEED[seg])

    train_df = load_segment_csv(TRIPS_DIR, seg, "train")
    oos_df   = load_segment_csv(TRIPS_DIR, seg, "oos")

    seed_tr   = BASE_SEED + 1000 * SEG_SEED[seg] + 10
    seed_te   = BASE_SEED + 1000 * SEG_SEED[seg] + 20
    seed_full = FULL_EVAL_SEED + 1000 * SEG_SEED[seg]

    emu_tr, y_tr, w_tr, train_df2, destset_tr, origidx_tr, destidx_tr = build_design_emu_only(
        train_df, tz_ids, emu_mat, zone_to_idx, k=K, seed=seed_tr
    )
    emu_te, y_te, w_te, oos_df2, destset_te, origidx_te, destidx_te = build_design_emu_only(
        oos_df, tz_ids, emu_mat, zone_to_idx, k=K, seed=seed_te
    )

    print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
    print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

    alpha_hat = train_mnl_emu_only(
        emu_tr, y_tr, w_tr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
    )

    met_tr = eval_sampled_metrics_emu_only(emu_tr, w_tr, alpha_hat, topKs=TOPKS_SAMPLED, want_ff=True)
    met_te = eval_sampled_metrics_emu_only(emu_te, w_te, alpha_hat, topKs=TOPKS_SAMPLED, want_ff=True)

    rows_sampled.append({
        "segment": seg,
        "split": "train",
        "alpha_emu": float(alpha_hat),
        "K": K,
        **met_tr
    })
    rows_sampled.append({
        "segment": seg,
        "split": "oos",
        "alpha_emu": float(alpha_hat),
        "K": K,
        **met_te
    })

    print("[SAMPLED OOS ]",
          f"NLL_sum={met_te['NLL_sum']:.2f} NLL_avg={met_te['NLL_avg']:.4f} "
          f"MRR={met_te['MRR']:.4f} Top5={met_te['Top5']:.3f} Top10={met_te['Top10']:.3f} FF={met_te['FF']:.4f}")

    max_n = FULL_EVAL_MAX.get(seg, None)
    full_out, share_pack, dist_pack, trip_distance_df = full_eval_and_shares_oos_subset_emu_only(
        oos_df=oos_df2,
        tz_ids=tz_ids,
        emu_mat=emu_mat,
        dist_mat=dist_mat,
        zone_to_idx=zone_to_idx,
        idx_to_zone=idx_to_zone,
        alpha_hat=alpha_hat,
        max_n=max_n,
        seed=seed_full,
        chunk=FULL_EVAL_CHUNK,
        topKs=TOPKS_FULL,
    )

    rows_full.append({
        "segment": seg,
        "alpha_emu": float(alpha_hat),
        "K": K,
        **full_out
    })

    print("[FULL OOS   ]",
          f"NLL_full_avg={full_out['NLL_full_avg']:.4f} MRR_full={full_out['MRR_full']:.4f} "
          f"FF_full={full_out['FF_full']:.6f} "
          f"R2_full={full_out['McFadden_R2_full']:.4f} Top5_full={full_out['Top5_full']:.3f} Top10_full={full_out['Top10_full']:.3f}")
    print("[SHARES FULL]",
          f"Pear={full_out['Pearson_share']:.4f} Spear={full_out['Spearman_share']:.4f} "
          f"MAE={full_out['MAE_share']:.6f} RMSE={full_out['RMSE_share']:.6f} JS={full_out['JS_share']:.6f}")

    obs_dist, exp_dist = dist_pack
    if (obs_dist is not None) and (exp_dist is not None):
        png = os.path.join(PLOT_DIR, f"distance_plausibility_{seg}.png")
        plot_distance_plausibility(obs_dist, exp_dist, seg, png)
        print(f"[PLOT] wrote {png}")

        if trip_distance_df is not None:
            dist_csv = os.path.join(DIST_OOS_DIR, f"distance_triplevel_{seg}.csv")
            trip_distance_df.to_csv(dist_csv, index=False)
            print(f"[CSV ] wrote {dist_csv}")

    # Hessian-based alpha SE
    se_alpha = approx_hessian_se_alpha_only(
        emu_set=emu_tr,
        y=y_tr,
        w=w_tr,
        alpha_hat=alpha_hat,
        max_n=SE_MAX_TRAIN.get(seg, 20000),
        seed=SE_SEED + 1000 * SEG_SEED[seg],
        ridge=1e-8,
    )

    zval = alpha_hat / (se_alpha + 1e-12)
    pval = p_value_two_sided_z(zval)
    rows_coef.append({
        "segment": seg,
        "param": "alpha_emu",
        "coef": float(alpha_hat),
        "se": float(se_alpha),
        "z": float(zval),
        "p": float(pval),
        "stars": stars(pval),
    })

    # TZ outputs
    if share_pack is not None:
        tz_out = tz[["npvm_id", "geometry"]].copy()
        tz_out = tz_out.set_index("npvm_id").loc[share_pack["tz_ids"]].reset_index()

        tz_out[f"S_obs_{seg}"]  = share_pack["S_obs"]
        tz_out[f"S_pred_{seg}"] = share_pack["S_pred"]

        png = os.path.join(PLOT_DIR, f"map_S_pred_{seg}.png")
        plot_share_map(tz_out, f"S_pred_{seg}", seg, png)
        print(f"[PLOT] wrote {png}")

        layer_name = f"TZ_{seg}"
        tz_out.to_file(GPKG_OUT, layer=layer_name, driver="GPKG")
        print(f"[GPKG] wrote layer {layer_name} -> {GPKG_OUT}")

# save tables
df_sampled = pd.DataFrame(rows_sampled)
df_full    = pd.DataFrame(rows_full)
df_coef    = pd.DataFrame(rows_coef)

p1 = os.path.join(OUT_DIR, "metrics_sampled.csv")
p2 = os.path.join(OUT_DIR, "metrics_full_and_shares.csv")
p3 = os.path.join(OUT_DIR, "coef_table.csv")

df_sampled.to_csv(p1, index=False)
df_full.to_csv(p2, index=False)
df_coef.to_csv(p3, index=False)

print("\n" + "=" * 110)
print("[OK] wrote", p1)
print("[OK] wrote", p2)
print("[OK] wrote", p3)
print("[OK] wrote", GPKG_OUT)
print("[OK] plots in", PLOT_DIR)
print("[OK] trip-level distance CSVs in", DIST_OOS_DIR)
print("=" * 110)

print("\nSAMPLED OOS METRICS (weighted):")
print(
    df_sampled[df_sampled["split"] == "oos"][["segment", "NLL_sum", "NLL_avg", "MRR", "Top5", "Top10", "FF", "N", "J", "sum_WP"]]
    .sort_values("segment")
    .to_string(index=False)
)

print("\nFULL OOS METRICS + SHARE DIAGNOSTICS:")
keep_cols = [
    "segment", "NLL_full_sum", "NLL_full_avg", "MRR_full", "FF_full", "McFadden_R2_full", "Top5_full", "Top10_full",
    "Pearson_share", "Spearman_share", "MAE_share", "RMSE_share", "JS_share", "N_full_eval", "WP_full_eval", "J_full",
    "dist_obs_mean_km", "dist_pred_mean_km", "dist_mean_diff_km", "dist_obs_p50_km", "dist_pred_p50_km", "dist_obs_p90_km", "dist_pred_p90_km",
]
cols_exist = [c for c in keep_cols if c in df_full.columns]
print(df_full[cols_exist].sort_values("segment").to_string(index=False))

## SIMBA MOBi baseline specificiation

In [ ]:
# ============================================================
# BASELINE ESTIMATION + FULL REPORT (withLIE, byPerson)
#   - EMU + SIMBA vars (3): pop_total, schl_enr_3, visit_L
#   - join TZ gpkg (npvm_id) with acc_2023.csv (zone_id)
#   - apply log1p to SIMBA vars, then global z-score on FULL TZ universe
#
# Aligned with the final baseline script:
#   - sampled MNL training (chosen + K non-chosen), uniform sampling
#   - sampled OOS metrics (NLL, TopK, MRR, FF)
#   - FULL denominator evaluation (chunked) + shares diagnostics
#   - FULL FF_full
#   - distance plausibility via avg_distances_ready (network average distance)
#   - SAME bins for observed vs predicted histograms
#   - means shown in legend/title
#   - trip-level OOS distance csv export
#   - Hessian-based SEs on a train subset
#   - outputs: csv tables + tz_outputs.gpkg + plots
# ============================================================

import os
import math
import random
import numpy as np
import pandas as pd
import geopandas as gpd
import tables as tb

import torch
import openmatrix as omx
from pathlib import Path

import matplotlib.pyplot as plt


# -----------------------------
# CONFIG (withLIE + byPerson)
# -----------------------------
SEGMENTS  = ["YS", "OS", "YL", "OL"]
SEG_SEED  = {"YS": 1, "OS": 2, "YL": 3, "OL": 4}  # deterministic

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"
DIST_COL  = "dist_km"  # observed trip distance from microcensus

TRIPS_DIR = "Trips/byPerson"  # WITH Liechtenstein

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

# SIMBA inputs
ACC_CSV        = "TravelCost/accessibility/2023/acc_2023.csv"
ACC_JOIN_KEY   = "zone_id"   # in acc.csv
TZ_JOIN_KEY    = "npvm_id"   # in TZ gpkg
SIMBA_VARS_RAW = ["pop_total", "schl_enr_3", "visit_L"]

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")

# Impedance (EMU)
UTIL_OMX      = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME  = "utility_emu"

# Distance proxy for plausibility (average car-network distance)
DIST_OMX      = READY_DIR / "distance_avg_2023_ready.omx"
DIST_MAT_NAME = "avg_distances_ready"

MAP_NAME = "NO"

OUT_DIR  = "ModelRuns/SIMBA_Only3Vars"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DIST_OOS_DIR = os.path.join(OUT_DIR, "distance_oos_triplevel")
os.makedirs(DIST_OOS_DIR, exist_ok=True)

# sampled training
K = 1000
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# full eval
FULL_EVAL_MAX = {"YS": 20000, "OS": 8000, "YL": 8000, "OL": 8000}
FULL_EVAL_SEED  = 777
FULL_EVAL_CHUNK = 256

# Hessian SEs on train subset
SE_MAX_TRAIN = {"YS": 40000, "OS": 25000, "YL": 12000, "OL": 6000}
SE_SEED  = 2024

BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

TOPKS_SAMPLED = (5, 10)
TOPKS_FULL    = (5, 10)


# ============================================================
# Reproducibility
# ============================================================
def set_all_seeds(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Small helpers
# ============================================================
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def p_value_two_sided_z(z):
    return 2.0 * (1.0 - norm_cdf(abs(float(z))))

def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.1:
        return "."
    return ""


# ============================================================
# IO helpers
# ============================================================
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()

    if DIST_COL in df.columns:
        df[DIST_COL] = pd.to_numeric(df[DIST_COL], errors="coerce")

    return df

def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone


# ============================================================
# Preprocess helpers
# ============================================================
def log1p_nonneg(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    x = x.clip(lower=0)
    return np.log1p(x)

def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    """
    Z-standardize on the FULL TZ universe.
    Returns:
      tz_feat (with added z columns), z_cols, mu, sd
    """
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd


# ============================================================
# Sampling + design matrix
# ============================================================
def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray, k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0]  = c
        dest_set[i, 1:] = alts
    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    """
    Builds sampled choice-set tensors:
      emu_set: (N, K+1)
      X_set:   (N, K+1, P)
      y:       (N,) always 0
      w:       (N,)
    """
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]
    dest_set2 = dest_set[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
        dest_set2,
        orig_idx2,
        dest_idx2,
    )


# ============================================================
# Model
# ============================================================
def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)
            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum()

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted-avg NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(5,10), want_ff=True):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        P = torch.exp(logP)

        chosen_logp = logP[:, 0]
        chosen_p    = P[:, 0]

        sum_w = float(w.sum().cpu())
        nll_sum = float((-(w * chosen_logp)).sum().cpu())
        nll_avg = nll_sum / (sum_w + 1e-9)

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))

        out = {
            "NLL_sum": nll_sum,
            "NLL_avg": nll_avg,
            "MRR": mrr,
            "sum_WP": sum_w,
            "N": int(V.shape[0]),
            "J": int(V.shape[1]),
        }

        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))

        if want_ff:
            out["FF"] = float((w * chosen_p).sum().cpu() / (sum_w + 1e-9))

    return out


# ============================================================
# FULL-choice-set evaluation
# ============================================================
def full_eval_and_shares_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    dist_mat: np.ndarray | None,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(5,10),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w    = df[WP_COL].to_numpy(dtype=np.float64)

    has_dist = (DIST_COL in df.columns) and df[DIST_COL].notna().any()
    obs_dist = df[DIST_COL].to_numpy(dtype=np.float64) if has_dist else None

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)

    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)
    if has_dist:
        good = good & np.isfinite(obs_dist) & (obs_dist >= 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w        = w[good]
    if has_dist:
        obs_dist = obs_dist[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}, None, (None, None), None

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass  = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    ll_model = 0.0
    mrr_num = 0.0
    ff_num = 0.0
    top_num = {k: 0.0 for k in topKs}
    ranks   = np.empty(len(df), dtype=np.int64)

    exp_dist = np.zeros(len(df), dtype=np.float64) if (dist_mat is not None and has_dist) else None

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse
        p_chosen = math.exp(logp_chosen)

        nll_num  += wn * (-logp_chosen)
        ll_model += wn * logp_chosen
        ff_num   += wn * p_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        if exp_dist is not None:
            expd = 0.0

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

            if exp_dist is not None:
                dist_blk = dist_mat[o, j0:j1].astype(np.float64)
                expd += float(np.sum(Pblk * dist_blk))

        if exp_dist is not None:
            exp_dist[n] = expd

    nll_full = float(nll_num)
    nll_full_avg = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    ff_full  = float(ff_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs  = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)

    pear  = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="pearson")) if len(S_obs) > 2 else np.nan
    spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    mae   = float(np.mean(np.abs(S_obs - S_pred)))
    rmse  = float(np.sqrt(np.mean((S_obs - S_pred) ** 2)))
    js    = float(js_div(S_obs, S_pred))

    dist_summary = {}
    if exp_dist is not None:
        wnorm = w / (sum_w + 1e-12)
        obs_mean  = float(np.sum(wnorm * obs_dist))
        pred_mean = float(np.sum(wnorm * exp_dist))
        dist_summary = {
            "dist_obs_mean_km": obs_mean,
            "dist_pred_mean_km": pred_mean,
            "dist_mean_diff_km": float(pred_mean - obs_mean),
            "dist_obs_p50_km": float(np.quantile(obs_dist, 0.50)),
            "dist_pred_p50_km": float(np.quantile(exp_dist, 0.50)),
            "dist_obs_p90_km": float(np.quantile(obs_dist, 0.90)),
            "dist_pred_p90_km": float(np.quantile(exp_dist, 0.90)),
        }

    out = {
        "NLL_full_sum": nll_full,
        "NLL_full_avg": nll_full_avg,
        "MRR_full": mrr_full,
        "FF_full": ff_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "Pearson_share": pear,
        "Spearman_share": spear,
        "MAE_share": mae,
        "RMSE_share": rmse,
        "JS_share": js,
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    out.update(dist_summary)

    trip_distance_df = None
    if exp_dist is not None:
        trip_distance_df = pd.DataFrame({
            "orig_zone": df[ORIG_COL].to_numpy(dtype=int),
            "dest_zone_obs": df[DEST_COL].to_numpy(dtype=int),
            "WP": w.astype(np.float64),
            "dist_obs_km": obs_dist.astype(np.float64),
            "dist_exp_km": exp_dist.astype(np.float64),
        })

    return out, {
        "tz_ids": all_zone_ids,
        "S_obs": S_obs,
        "S_pred": S_pred,
        "Aj": Aj.astype(np.float64),
    }, (obs_dist, exp_dist), trip_distance_df


# ============================================================
# Hessian-based SEs
# ============================================================
def approx_hessian_se(
    emu_set: torch.Tensor,
    X_set: torch.Tensor,
    y: torch.Tensor,
    w: torch.Tensor,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int,
    seed: int,
    ridge: float = 1e-8,
):
    """
    Approximate standard errors using the inverse Hessian of the
    weighted negative log-likelihood, evaluated at the estimated
    coefficients (alpha_hat, beta_hat).
    """
    N = emu_set.shape[0]
    rng = np.random.default_rng(seed)

    if N > max_n:
        idx = rng.choice(N, size=max_n, replace=False)
        idx = torch.tensor(idx, dtype=torch.long, device=DEVICE)
        emu = emu_set[idx]
        X   = X_set[idx]
        yy  = y[idx]
        ww  = w[idx]
    else:
        emu, X, yy, ww = emu_set, X_set, y, w

    sum_w = float(ww.sum().detach().cpu())
    Pdim = X.shape[2]

    alpha = torch.tensor([alpha_hat], dtype=DTYPE, device=DEVICE, requires_grad=True)
    beta  = torch.tensor(beta_hat, dtype=DTYPE, device=DEVICE, requires_grad=True)

    def loss_avg(a, b):
        V = a * emu + torch.einsum("bjp,p->bj", X, b)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        ll = logP[torch.arange(V.shape[0], device=DEVICE), yy]
        return (-(ww * ll)).sum() / (ww.sum() + 1e-12)

    L = loss_avg(alpha, beta)
    g = torch.autograd.grad(L, [alpha, beta], create_graph=True)
    g_vec = torch.cat([g[0].reshape(1), g[1].reshape(-1)], dim=0)

    H_avg = torch.zeros((1 + Pdim, 1 + Pdim), dtype=torch.float64, device=DEVICE)
    for i in range(1 + Pdim):
        gi = g_vec[i]
        hi = torch.autograd.grad(gi, [alpha, beta], retain_graph=True)
        hi_vec = torch.cat([hi[0].reshape(1), hi[1].reshape(-1)], dim=0)
        H_avg[i, :] = hi_vec.detach().to(torch.float64)

    H_avg = H_avg.cpu().numpy()
    H_sum = sum_w * H_avg

    H_sum = 0.5 * (H_sum + H_sum.T)
    H_sum = H_sum + ridge * np.eye(H_sum.shape[0], dtype=np.float64)

    Vcov = np.linalg.pinv(H_sum)
    se = np.sqrt(np.clip(np.diag(Vcov), 0.0, np.inf))
    return se


# ============================================================
# Plots
# ============================================================
def plot_distance_plausibility(obs_dist, exp_dist, seg, out_png):
    obs_dist = np.asarray(obs_dist, dtype=np.float64)
    exp_dist = np.asarray(exp_dist, dtype=np.float64)

    finite = np.isfinite(obs_dist) & np.isfinite(exp_dist)
    obs_dist = obs_dist[finite]
    exp_dist = exp_dist[finite]

    if len(obs_dist) == 0:
        return

    obs_mean = float(np.mean(obs_dist))
    exp_mean = float(np.mean(exp_dist))

    xmax = float(max(np.max(obs_dist), np.max(exp_dist)))
    bins = np.linspace(0.0, xmax, 51)  # same 50 bins

    plt.figure()
    plt.hist(obs_dist, bins=bins, density=True, alpha=0.6,
             label=f"Observed dist_km (mean={obs_mean:.2f})")
    plt.hist(exp_dist, bins=bins, density=True, alpha=0.6,
             label=f"Predicted E[distance] (mean={exp_mean:.2f})")
    plt.xlabel("Distance (km)")
    plt.ylabel("Density")
    plt.title(f"Distance plausibility | {seg} | SIMBA-3vars | obs mean={obs_mean:.2f}, pred mean={exp_mean:.2f}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_share_maps(tz_gdf, col, seg, out_png, title):
    plt.figure()
    ax = tz_gdf.plot(column=col, legend=True)
    ax.set_axis_off()
    plt.title(f"{title} | {seg}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()


# ============================================================
# MAIN
# ============================================================
set_all_seeds(BASE_SEED)

print("[LOAD] TZ layer...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
if TZ_JOIN_KEY not in tz.columns:
    raise ValueError(f"TZ layer missing '{TZ_JOIN_KEY}'. Columns: {list(tz.columns)}")

tz[TZ_JOIN_KEY] = pd.to_numeric(tz[TZ_JOIN_KEY], errors="coerce")
tz = tz.dropna(subset=[TZ_JOIN_KEY]).copy()
tz[TZ_JOIN_KEY] = tz[TZ_JOIN_KEY].astype(int)

print("[LOAD] ACC csv (sep=';')...")
acc = pd.read_csv(ACC_CSV, sep=";", low_memory=False)

if ACC_JOIN_KEY not in acc.columns:
    raise ValueError(f"ACC csv missing '{ACC_JOIN_KEY}'. Columns: {list(acc.columns)}")

need_acc = [ACC_JOIN_KEY] + SIMBA_VARS_RAW
miss_acc = [c for c in need_acc if c not in acc.columns]
if miss_acc:
    raise ValueError(f"ACC csv missing columns: {miss_acc}")

acc = acc[need_acc].copy()
acc[ACC_JOIN_KEY] = pd.to_numeric(acc[ACC_JOIN_KEY], errors="coerce")
acc = acc.dropna(subset=[ACC_JOIN_KEY]).copy()
acc[ACC_JOIN_KEY] = acc[ACC_JOIN_KEY].astype(int)

for v in SIMBA_VARS_RAW:
    acc[v] = pd.to_numeric(acc[v], errors="coerce")

print("[JOIN] tz (npvm_id) <- acc (zone_id)...")
tz_feat = tz.merge(acc, how="left", left_on=TZ_JOIN_KEY, right_on=ACC_JOIN_KEY)

na_join = tz_feat[SIMBA_VARS_RAW].isna().any(axis=1).sum()
print(f"[JOIN] NA in any SIMBA var after join: {na_join:,}/{len(tz_feat):,}")

print("[PREP] log1p(SIMBA vars)...")
SIMBA_VARS_LOG1P = [f"{v}_log1p" for v in SIMBA_VARS_RAW]
for v, v2 in zip(SIMBA_VARS_RAW, SIMBA_VARS_LOG1P):
    tz_feat[v2] = log1p_nonneg(tz_feat[v])

print("[PREP] Global z-standardize log1p SIMBA vars on FULL TZ universe...")
tz_feat, Z_COLS, MU_ALL, SD_ALL = standardize_all_zones(tz_feat, SIMBA_VARS_LOG1P, suffix="_z")
print("[INFO] Using Z_COLS:", Z_COLS)

if "npvm_id" not in tz_feat.columns:
    raise ValueError("Expected 'npvm_id' in tz_feat")

print("[LOAD] EMU matrix + mapping...")
emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

print("[LOAD] Average distance matrix (for distance plausibility)...")
dist_mat, zone_to_idx_dist, idx_to_zone_dist = load_matrix_and_mapping(DIST_OMX, DIST_MAT_NAME, MAP_NAME)
if len(idx_to_zone_dist) != len(idx_to_zone) or not np.all(idx_to_zone_dist == idx_to_zone):
    print("[WARN] Distance OMX mapping differs from utilities OMX mapping. Distance plausibility may be misaligned.")

rows_sampled = []
rows_full    = []
rows_coef    = []

GPKG_OUT = os.path.join(OUT_DIR, "tz_outputs.gpkg")
if os.path.exists(GPKG_OUT):
    os.remove(GPKG_OUT)

for seg in SEGMENTS:
    print("\n" + "#" * 120)
    print(f"# SEGMENT = {seg} | SIMBA-3vars | K={K} | epochs={EPOCHS}")
    print("#" * 120)

    set_all_seeds(BASE_SEED + 1000 * SEG_SEED[seg])

    train_df = load_segment_csv(TRIPS_DIR, seg, "train")
    oos_df   = load_segment_csv(TRIPS_DIR, seg, "oos")

    seed_tr   = BASE_SEED + 1000 * SEG_SEED[seg] + 10
    seed_te   = BASE_SEED + 1000 * SEG_SEED[seg] + 20
    seed_full = FULL_EVAL_SEED + 1000 * SEG_SEED[seg]

    emu_tr, X_tr, y_tr, w_tr, train_df2, destset_tr, origidx_tr, destidx_tr = build_design(
        train_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_tr
    )
    emu_te, X_te, y_te, w_te, oos_df2, destset_te, origidx_te, destidx_te = build_design(
        oos_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_te
    )

    print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
    print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

    alpha_hat, beta_hat = train_mnl(
        emu_tr, X_tr, y_tr, w_tr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
    )

    met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)
    met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)

    rows_sampled.append({"segment": seg, "split": "train", "alpha_emu": float(alpha_hat), "K": K, **met_tr})
    rows_sampled.append({"segment": seg, "split": "oos",   "alpha_emu": float(alpha_hat), "K": K, **met_te})

    print(
        "[SAMPLED OOS ]",
        f"NLL_sum={met_te['NLL_sum']:.2f} NLL_avg={met_te['NLL_avg']:.4f} "
        f"MRR={met_te['MRR']:.4f} Top5={met_te['Top5']:.3f} Top10={met_te['Top10']:.3f} FF={met_te['FF']:.4f}"
    )

    max_n = FULL_EVAL_MAX.get(seg, None)
    full_out, share_pack, dist_pack, trip_distance_df = full_eval_and_shares_oos_subset(
        oos_df=oos_df2,
        tz_feat=tz_feat,
        z_cols=Z_COLS,
        emu_mat=emu_mat,
        dist_mat=dist_mat,
        zone_to_idx=zone_to_idx,
        idx_to_zone=idx_to_zone,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=max_n,
        seed=seed_full,
        chunk=FULL_EVAL_CHUNK,
        topKs=TOPKS_FULL,
    )

    rows_full.append({"segment": seg, "alpha_emu": float(alpha_hat), "K": K, **full_out})

    print(
        "[FULL OOS   ]",
        f"NLL_full_avg={full_out['NLL_full_avg']:.4f} MRR_full={full_out['MRR_full']:.4f} "
        f"FF_full={full_out['FF_full']:.6f} "
        f"R2_full={full_out['McFadden_R2_full']:.4f} Top5_full={full_out['Top5_full']:.3f} Top10_full={full_out['Top10_full']:.3f}"
    )
    print(
        "[SHARES FULL]",
        f"Pear={full_out['Pearson_share']:.4f} Spear={full_out['Spearman_share']:.4f} "
        f"MAE={full_out['MAE_share']:.6f} RMSE={full_out['RMSE_share']:.6f} JS={full_out['JS_share']:.6f}"
    )

    obs_dist, exp_dist = dist_pack
    if (obs_dist is not None) and (exp_dist is not None):
        png = os.path.join(PLOT_DIR, f"distance_plausibility_{seg}.png")
        plot_distance_plausibility(obs_dist, exp_dist, seg, png)
        print(f"[PLOT] wrote {png}")

        if trip_distance_df is not None:
            dist_csv = os.path.join(DIST_OOS_DIR, f"distance_triplevel_{seg}.csv")
            trip_distance_df.to_csv(dist_csv, index=False)
            print(f"[CSV ] wrote {dist_csv}")

    se_vec = approx_hessian_se(
        emu_set=emu_tr,
        X_set=X_tr,
        y=y_tr,
        w=w_tr,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=SE_MAX_TRAIN.get(seg, 20000),
        seed=SE_SEED + 1000 * SEG_SEED[seg],
        ridge=1e-8,
    )

    se_alpha = float(se_vec[0])
    se_beta  = se_vec[1:]

    zval = alpha_hat / (se_alpha + 1e-12)
    pval = p_value_two_sided_z(zval)
    rows_coef.append({
        "segment": seg,
        "param": "alpha_emu",
        "coef": float(alpha_hat),
        "se": float(se_alpha),
        "z": float(zval),
        "p": float(pval),
        "stars": stars(pval),
    })

    for name, b, s in zip(Z_COLS, beta_hat.tolist(), se_beta.tolist()):
        zval = b / (s + 1e-12)
        pval = p_value_two_sided_z(zval)
        rows_coef.append({
            "segment": seg,
            "param": name,
            "coef": float(b),
            "se": float(s),
            "z": float(zval),
            "p": float(pval),
            "stars": stars(pval),
        })

    if share_pack is not None:
        tz_ids = share_pack["tz_ids"]
        S_obs  = share_pack["S_obs"]
        S_pred = share_pack["S_pred"]
        Aj     = share_pack["Aj"]

        tz_out = tz_feat[["npvm_id", "geometry"]].copy()
        tz_out = tz_out.set_index("npvm_id").loc[tz_ids].reset_index()

        tz_out[f"A_attr_{seg}"] = Aj
        tz_out[f"S_obs_{seg}"]  = S_obs
        tz_out[f"S_pred_{seg}"] = S_pred

        png1 = os.path.join(PLOT_DIR, f"map_S_pred_{seg}.png")
        png2 = os.path.join(PLOT_DIR, f"map_A_attr_{seg}.png")
        plot_share_maps(tz_out, f"S_pred_{seg}", seg, png1, "Predicted destination share (OOS subset, full denom)")
        plot_share_maps(tz_out, f"A_attr_{seg}", seg, png2, "Attractivity index A_j = X_j beta (SIMBA z vars)")
        print(f"[PLOT] wrote {png1}")
        print(f"[PLOT] wrote {png2}")

        layer_name = f"TZ_{seg}"
        tz_out.to_file(GPKG_OUT, layer=layer_name, driver="GPKG")
        print(f"[GPKG] wrote layer {layer_name} -> {GPKG_OUT}")


# ============================================================
# Save summary tables
# ============================================================
df_sampled = pd.DataFrame(rows_sampled)
df_full    = pd.DataFrame(rows_full)
df_coef    = pd.DataFrame(rows_coef)

p1 = os.path.join(OUT_DIR, "metrics_sampled.csv")
p2 = os.path.join(OUT_DIR, "metrics_full_and_shares.csv")
p3 = os.path.join(OUT_DIR, "coef_table.csv")

df_sampled.to_csv(p1, index=False)
df_full.to_csv(p2, index=False)
df_coef.to_csv(p3, index=False)

print("\n" + "=" * 110)
print("[OK] wrote", p1)
print("[OK] wrote", p2)
print("[OK] wrote", p3)
print("[OK] wrote", GPKG_OUT)
print("[OK] plots in", PLOT_DIR)
print("[OK] trip-level distance CSVs in", DIST_OOS_DIR)
print("=" * 110)

print("\nSAMPLED OOS METRICS (weighted):")
print(
    df_sampled[df_sampled["split"] == "oos"][
        ["segment", "NLL_sum", "NLL_avg", "MRR", "Top5", "Top10", "FF", "N", "J", "sum_WP"]
    ]
    .sort_values("segment")
    .to_string(index=False)
)

print("\nFULL OOS METRICS + SHARE DIAGNOSTICS:")
keep_cols = [
    "segment", "NLL_full_sum", "NLL_full_avg", "MRR_full", "FF_full", "McFadden_R2_full", "Top5_full", "Top10_full",
    "Pearson_share", "Spearman_share", "MAE_share", "RMSE_share", "JS_share",
    "N_full_eval", "WP_full_eval", "J_full",
    "dist_obs_mean_km", "dist_pred_mean_km", "dist_mean_diff_km",
    "dist_obs_p50_km", "dist_pred_p50_km", "dist_obs_p90_km", "dist_pred_p90_km",
]
cols_exist = [c for c in keep_cols if c in df_full.columns]
print(df_full[cols_exist].sort_values("segment").to_string(index=False))

## No segmentation specification

In [ ]:
# ============================================================
# BASELINE ESTIMATION + FULL REPORT (withLIE, byPerson)
#   - NOT segmented by age in estimation structure
#   - 3 pooled models:
#       S = YS + OS
#       L = YL + OL
#       F = YS + OS + YL + OL
#
# DATASET BUILD RULE (per split train/oos, per model):
#   - direct concatenation of source segments
#   - NO balancing / NO downsampling / NO oversampling
#
# Aligned with the final baseline script:
#   - sampled MNL training (chosen + K non-chosen), uniform sampling
#   - sampled OOS metrics: NLL, TopK, MRR, FF
#   - FULL denominator evaluation (chunked) + shares diagnostics
#   - FULL FF_full
#   - distance plausibility via avg_distances_ready (network average distance)
#   - SAME bins for observed vs predicted histograms
#   - means shown in legend/title
#   - trip-level OOS distance csv export
#   - Hessian-based SEs on a train subset
#   - outputs: csv tables + tz_outputs.gpkg + plots
# ============================================================

import os
import math
import random
import numpy as np
import pandas as pd
import geopandas as gpd
import tables as tb

import torch
import openmatrix as omx
from pathlib import Path

import matplotlib.pyplot as plt


# -----------------------------
# CONFIG
# -----------------------------
MODELS = ["S", "L", "F"]
MODEL_TO_SOURCE_SEGS = {
    "S": ("YS", "OS"),
    "L": ("YL", "OL"),
    "F": ("YS", "OS", "YL", "OL"),
}
MODEL_SEED = {"S": 11, "L": 22, "F": 33}

ORIG_COL  = "orig_zone"
DEST_COL  = "dest_zone"
WP_COL    = "WP"
DIST_COL  = "dist_km"  # observed trip distance from microcensus

TRIPS_DIR = "Trips/byPerson"

TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")

# Impedance (EMU)
UTIL_OMX      = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME  = "utility_emu"

# Distance proxy for plausibility (average car-network distance)
DIST_OMX      = READY_DIR / "distance_avg_2023_ready.omx"
DIST_MAT_NAME = "avg_distances_ready"

MAP_NAME = "NO"

OUT_DIR  = "ModelRuns/Baseline_NoAge_3Models"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DIST_OOS_DIR = os.path.join(OUT_DIR, "distance_oos_triplevel")
os.makedirs(DIST_OOS_DIR, exist_ok=True)

# Core baseline decisions
K = 1000
EPOCHS = 50
BATCH_SIZE = 2048
LR = 0.03
WEIGHT_DECAY = 0.0

# Full-choice-set evaluation subset sizes (OOS)
FULL_EVAL_MAX = {
    "S": 50000,
    "L": 10000,
    "F": 70000,
}
FULL_EVAL_SEED  = 777
FULL_EVAL_CHUNK = 256

# Hessian SEs on train subset
SE_MAX_TRAIN = {
    "S": 50000,
    "L": 20000,
    "F": 80000,
}
SE_SEED = 2024

BASE_SEED = 123

DEVICE = "cpu"
DTYPE  = torch.float32

TOPKS_SAMPLED = (5, 10)
TOPKS_FULL    = (5, 10)

# -----------------------------
# BASELINE FEATURE LIST (13)
# -----------------------------
BASE_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]


# ============================================================
# Reproducibility
# ============================================================
def set_all_seeds(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Small helpers
# ============================================================
def kl_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def js_div(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.maximum(p, 1e-300)
    q = np.maximum(q, 1e-300)
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def p_value_two_sided_z(z):
    return 2.0 * (1.0 - norm_cdf(abs(float(z))))

def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.1:
        return "."
    return ""


# ============================================================
# IO helpers
# ============================================================
def load_segment_csv(trips_dir: str, seg: str, split: str) -> pd.DataFrame:
    path = os.path.join(trips_dir, f"destinations_{seg}_{split}.csv")
    df = pd.read_csv(path, low_memory=False)

    for c in [ORIG_COL, DEST_COL, WP_COL]:
        if c not in df.columns:
            raise ValueError(f"{path}: missing '{c}'. Available: {list(df.columns)[:40]}")

    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()
    df[ORIG_COL] = pd.to_numeric(df[ORIG_COL], errors="coerce")
    df[DEST_COL] = pd.to_numeric(df[DEST_COL], errors="coerce")
    df[WP_COL]   = pd.to_numeric(df[WP_COL], errors="coerce")
    df = df.dropna(subset=[ORIG_COL, DEST_COL, WP_COL]).copy()

    df[ORIG_COL] = df[ORIG_COL].astype(int)
    df[DEST_COL] = df[DEST_COL].astype(int)
    df = df[df[WP_COL] > 0].copy()

    if DIST_COL in df.columns:
        df[DIST_COL] = pd.to_numeric(df[DIST_COL], errors="coerce")

    return df

def build_pooled_dataset(trips_dir: str, model: str, split: str) -> pd.DataFrame:
    segs = MODEL_TO_SOURCE_SEGS[model]
    dfs = []
    for seg in segs:
        dfi = load_segment_csv(trips_dir, seg, split)
        dfi["_src_seg"] = seg
        dfs.append(dfi)

    out = pd.concat(dfs, axis=0, ignore_index=True)
    print(f"[POOL BUILD] model={model} split={split} | sources={segs} | pooled N={len(out):,}")
    return out

def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}

    M = f[mat_name]
    print(f"[OMX] Loading {mat_name} into RAM... shape={M.shape} (float32)")
    mat = np.asarray(M[:, :], dtype=np.float32)

    f.close()
    return mat, zone_to_idx, idx_to_zone


# ============================================================
# Preprocess helpers
# ============================================================
def standardize_all_zones(tz_feat: pd.DataFrame, cols: list[str], suffix="_z"):
    mu = tz_feat[cols].mean(axis=0)
    sd = tz_feat[cols].std(axis=0).replace(0, np.nan)

    z_cols = []
    for c in cols:
        zc = f"{c}{suffix}"
        tz_feat[zc] = (tz_feat[c] - mu[c]) / sd[c]
        z_cols.append(zc)

    return tz_feat, z_cols, mu, sd


# ============================================================
# Sampling + design matrix
# ============================================================
def sample_choice_sets_unique(chosen_dest: np.ndarray, all_dests: np.ndarray,
                              k: int, rng: np.random.Generator):
    N = len(chosen_dest)
    dest_set = np.empty((N, k + 1), dtype=all_dests.dtype)
    for i in range(N):
        c = chosen_dest[i]
        pool = all_dests[all_dests != c]
        if k > len(pool):
            raise ValueError(f"K={k} too large for destination universe size {len(pool)+1}")
        alts = rng.choice(pool, size=k, replace=False)
        dest_set[i, 0]  = c
        dest_set[i, 1:] = alts
    return dest_set

def build_design(df: pd.DataFrame, tz_feat: pd.DataFrame, feat_cols_z: list[str],
                 emu_mat: np.ndarray, zone_to_idx: dict, k: int, seed: int):
    rng = np.random.default_rng(seed)

    chosen = df[DEST_COL].to_numpy(dtype=int)
    orig   = df[ORIG_COL].to_numpy(dtype=int)
    w      = df[WP_COL].to_numpy(dtype=np.float64)

    all_dests = tz_feat["npvm_id"].to_numpy(dtype=int)
    dest_set = sample_choice_sets_unique(chosen, all_dests, k=k, rng=rng)

    feat_map = tz_feat.set_index("npvm_id")[feat_cols_z]

    N = len(df)
    Pdim = len(feat_cols_z)
    X = np.empty((N, k + 1, Pdim), dtype=np.float32)

    flat = dest_set.ravel()
    for p, col in enumerate(feat_cols_z):
        X[:, :, p] = feat_map.loc[flat, col].to_numpy(dtype=np.float32).reshape(N, k + 1)

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in flat], dtype=int).reshape(N, k + 1)

    good = (orig_idx >= 0) & (dest_idx.min(axis=1) >= 0) & np.isfinite(w) & (w > 0)
    if good.mean() < 1.0:
        print(f"[INFO] Dropping {(~good).sum()} rows due to missing mapping / bad weights.")

    df2 = df.loc[good].copy()
    w2 = w[good]
    X2 = X[good]
    orig_idx2 = orig_idx[good]
    dest_idx2 = dest_idx[good]
    dest_set2 = dest_set[good]

    emu_set = emu_mat[orig_idx2[:, None], dest_idx2].astype(np.float32)
    y = np.zeros(len(df2), dtype=np.int64)

    return (
        torch.tensor(emu_set, dtype=DTYPE, device=DEVICE),
        torch.tensor(X2, dtype=DTYPE, device=DEVICE),
        torch.tensor(y, dtype=torch.long, device=DEVICE),
        torch.tensor(w2, dtype=DTYPE, device=DEVICE),
        df2,
        dest_set2,
        orig_idx2,
        dest_idx2,
    )


# ============================================================
# Model
# ============================================================
def train_mnl(emu_set, X_set, y, w, epochs=50, lr=0.03, batch_size=2048, weight_decay=0.0):
    N, _ = emu_set.shape
    Pdim = X_set.shape[2]

    alpha = torch.zeros(1, device=DEVICE, dtype=DTYPE, requires_grad=True)
    beta  = torch.zeros(Pdim, device=DEVICE, dtype=DTYPE, requires_grad=True)

    opt = torch.optim.Adam([alpha, beta], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10)

    best = np.inf
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=DEVICE)
        tot_loss_num, tot_w = 0.0, 0.0

        for s in range(0, N, batch_size):
            b = perm[s:s + batch_size]
            emu_b = emu_set[b]
            X_b   = X_set[b]
            y_b   = y[b]
            w_b   = w[b]

            V = alpha * emu_b + torch.einsum("bjp,p->bj", X_b, beta)
            logP = V - torch.logsumexp(V, dim=1, keepdim=True)

            ll = logP[torch.arange(len(b), device=DEVICE), y_b]
            loss = -(w_b * ll).sum()

            opt.zero_grad()
            loss.backward()
            opt.step()

            tot_loss_num += float(loss.detach().cpu())
            tot_w += float(w_b.sum().detach().cpu())

        epoch_nll = tot_loss_num / (tot_w + 1e-9)
        sched.step(epoch_nll)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]["lr"]
            print(f"    epoch {ep:4d} | weighted-avg NLL={epoch_nll:.6f} | lr={lr_now:.5f}")

        if epoch_nll < best:
            best = epoch_nll
            best_state = (float(alpha.detach().cpu()), beta.detach().cpu().numpy().copy())

    return best_state

def eval_sampled_metrics(emu_set, X_set, w, alpha, beta, topKs=(5,10), want_ff=True):
    with torch.no_grad():
        beta_t  = torch.tensor(beta, dtype=DTYPE, device=DEVICE)
        alpha_t = torch.tensor([alpha], dtype=DTYPE, device=DEVICE)

        V = alpha_t * emu_set + torch.einsum("bjp,p->bj", X_set, beta_t)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        P = torch.exp(logP)

        chosen_logp = logP[:, 0]
        chosen_p    = P[:, 0]
        sum_w = float(w.sum().cpu())

        nll_sum = float((-(w * chosen_logp)).sum().cpu())
        nll_avg = nll_sum / (sum_w + 1e-9)

        order = torch.argsort(V, dim=1, descending=True)
        pos = (order == 0).nonzero(as_tuple=False)[:, 1]
        rank = (pos + 1).float()

        mrr = float((w * (1.0 / rank)).sum().cpu() / (sum_w + 1e-9))

        out = {
            "NLL_sum": nll_sum,
            "NLL_avg": nll_avg,
            "MRR": mrr,
            "sum_WP": sum_w,
            "N": int(V.shape[0]),
            "J": int(V.shape[1]),
        }
        for k in topKs:
            out[f"Top{k}"] = float((w * (rank <= k).float()).sum().cpu() / (sum_w + 1e-9))
        if want_ff:
            out["FF"] = float((w * chosen_p).sum().cpu() / (sum_w + 1e-9))
    return out


# ============================================================
# FULL-choice-set evaluation
# ============================================================
def full_eval_and_shares_oos_subset(
    oos_df: pd.DataFrame,
    tz_feat: pd.DataFrame,
    z_cols: list[str],
    emu_mat: np.ndarray,
    dist_mat: np.ndarray | None,
    zone_to_idx: dict,
    idx_to_zone: np.ndarray,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int | None,
    seed: int,
    chunk: int = 256,
    topKs=(5,10),
):
    if (max_n is not None) and (len(oos_df) > max_n):
        df = oos_df.sample(n=max_n, random_state=seed).copy()
    else:
        df = oos_df.copy()

    orig = df[ORIG_COL].to_numpy(dtype=int)
    dest = df[DEST_COL].to_numpy(dtype=int)
    w    = df[WP_COL].to_numpy(dtype=np.float64)

    has_dist = (DIST_COL in df.columns) and df[DIST_COL].notna().any()
    obs_dist = df[DIST_COL].to_numpy(dtype=np.float64) if has_dist else None

    orig_idx = np.array([zone_to_idx.get(int(z), -1) for z in orig], dtype=int)
    dest_idx = np.array([zone_to_idx.get(int(z), -1) for z in dest], dtype=int)

    good = (orig_idx >= 0) & (dest_idx >= 0) & np.isfinite(w) & (w > 0)
    if has_dist:
        good = good & np.isfinite(obs_dist) & (obs_dist >= 0)

    df = df.loc[good].copy()
    orig_idx = orig_idx[good]
    dest_idx = dest_idx[good]
    w        = w[good]
    if has_dist:
        obs_dist = obs_dist[good]

    sum_w = float(w.sum())
    if len(df) == 0:
        return {}, None, (None, None), None

    Msize = emu_mat.shape[0]
    alpha = float(alpha_hat)

    all_zone_ids = tz_feat["npvm_id"].to_numpy(dtype=int)
    Xz_all = tz_feat[z_cols].to_numpy(dtype=np.float32)
    Aj = Xz_all @ beta_hat.astype(np.float32)

    Aj_by_omxidx = np.zeros(Msize, dtype=np.float32)
    for zid, val in zip(all_zone_ids, Aj):
        jidx = zone_to_idx.get(int(zid), None)
        if jidx is not None and 0 <= jidx < Msize:
            Aj_by_omxidx[jidx] = float(val)

    tz_set = set(int(z) for z in all_zone_ids.tolist())
    tz_mask = np.array([int(z) in tz_set for z in idx_to_zone], dtype=bool)
    tz_positions = {int(z): i for i, z in enumerate(all_zone_ids.tolist())}

    pred_mass = np.zeros(len(all_zone_ids), dtype=np.float64)
    obs_mass  = (
        df.groupby(DEST_COL)[WP_COL].sum()
        .reindex(all_zone_ids, fill_value=0.0)
        .to_numpy(dtype=np.float64)
    )

    nll_num = 0.0
    ll_model = 0.0
    mrr_num = 0.0
    ff_num = 0.0
    top_num = {k: 0.0 for k in topKs}
    ranks   = np.empty(len(df), dtype=np.int64)

    exp_dist = np.zeros(len(df), dtype=np.float64) if (dist_mat is not None and has_dist) else None

    for n in range(len(df)):
        o = int(orig_idx[n])
        d = int(dest_idx[n])
        wn = float(w[n])

        Vchosen = alpha * float(emu_mat[o, d]) + float(Aj_by_omxidx[d])

        m = -np.inf
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            vbmax = float(np.max(Vblk))
            if vbmax > m:
                m = vbmax

        sumexp = 0.0
        better = 0
        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            V64 = Vblk.astype(np.float64)
            better += int(np.sum(V64 > Vchosen))
            sumexp += float(np.sum(np.exp(V64 - m)))

        lse = m + np.log(sumexp + 1e-300)
        logp_chosen = Vchosen - lse
        p_chosen = math.exp(logp_chosen)

        nll_num  += wn * (-logp_chosen)
        ll_model += wn * logp_chosen
        ff_num   += wn * p_chosen

        rank = better + 1
        ranks[n] = rank
        mrr_num += wn * (1.0 / float(rank))
        for k in topKs:
            if rank <= k:
                top_num[k] += wn

        if exp_dist is not None:
            expd = 0.0

        for j0 in range(0, Msize, chunk):
            j1 = min(Msize, j0 + chunk)
            Vblk = alpha * emu_mat[o, j0:j1].astype(np.float32) + Aj_by_omxidx[j0:j1]
            num = np.exp(Vblk.astype(np.float64) - m)
            Pblk = num / (sumexp + 1e-300)

            zids = idx_to_zone[j0:j1]
            msk = tz_mask[j0:j1]
            if np.any(msk):
                z_keep = zids[msk]
                p_keep = Pblk[msk]
                for zid, pj in zip(z_keep, p_keep):
                    pos = tz_positions.get(int(zid), None)
                    if pos is not None:
                        pred_mass[pos] += wn * float(pj)

            if exp_dist is not None:
                dist_blk = dist_mat[o, j0:j1].astype(np.float64)
                expd += float(np.sum(Pblk * dist_blk))

        if exp_dist is not None:
            exp_dist[n] = expd

    nll_full = float(nll_num)
    nll_full_avg = float(nll_num / (sum_w + 1e-9))
    mrr_full = float(mrr_num / (sum_w + 1e-9))
    ff_full = float(ff_num / (sum_w + 1e-9))
    rank_median_full = float(np.median(ranks))

    ll0 = sum_w * float(np.log(1.0 / Msize))
    r2_full = 1.0 - (ll_model / ll0) if ll0 != 0 else np.nan

    S_obs  = obs_mass / (obs_mass.sum() + 1e-12)
    S_pred = pred_mass / (pred_mass.sum() + 1e-12)

    pear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="pearson")) if len(S_obs) > 2 else np.nan
    spear = float(pd.Series(S_obs).corr(pd.Series(S_pred), method="spearman")) if len(S_obs) > 2 else np.nan
    mae  = float(np.mean(np.abs(S_obs - S_pred)))
    rmse = float(np.sqrt(np.mean((S_obs - S_pred) ** 2)))
    js   = float(js_div(S_obs, S_pred))

    dist_summary = {}
    if exp_dist is not None:
        wnorm = w / (sum_w + 1e-12)
        obs_mean = float(np.sum(wnorm * obs_dist))
        pred_mean = float(np.sum(wnorm * exp_dist))
        dist_summary = {
            "dist_obs_mean_km": obs_mean,
            "dist_pred_mean_km": pred_mean,
            "dist_mean_diff_km": float(pred_mean - obs_mean),
            "dist_obs_p50_km": float(np.quantile(obs_dist, 0.50)),
            "dist_pred_p50_km": float(np.quantile(exp_dist, 0.50)),
            "dist_obs_p90_km": float(np.quantile(obs_dist, 0.90)),
            "dist_pred_p90_km": float(np.quantile(exp_dist, 0.90)),
        }

    out = {
        "NLL_full_sum": nll_full,
        "NLL_full_avg": nll_full_avg,
        "MRR_full": mrr_full,
        "FF_full": ff_full,
        "McFadden_R2_full": float(r2_full),
        "rank_median_full": rank_median_full,
        "Pearson_share": pear,
        "Spearman_share": spear,
        "MAE_share": mae,
        "RMSE_share": rmse,
        "JS_share": js,
        "N_full_eval": int(len(df)),
        "J_full": int(Msize),
        "WP_full_eval": float(sum_w),
    }
    for k in topKs:
        out[f"Top{k}_full"] = float(top_num[k] / (sum_w + 1e-9))
    out.update(dist_summary)

    trip_distance_df = None
    if exp_dist is not None:
        trip_distance_df = pd.DataFrame({
            "orig_zone": df[ORIG_COL].to_numpy(dtype=int),
            "dest_zone_obs": df[DEST_COL].to_numpy(dtype=int),
            "WP": w.astype(np.float64),
            "dist_obs_km": obs_dist.astype(np.float64),
            "dist_exp_km": exp_dist.astype(np.float64),
        })

    return out, {
        "tz_ids": all_zone_ids,
        "S_obs": S_obs,
        "S_pred": S_pred,
        "Aj": Aj.astype(np.float64),
    }, (obs_dist, exp_dist), trip_distance_df


# ============================================================
# Hessian-based SEs
# ============================================================
def approx_hessian_se(
    emu_set: torch.Tensor,
    X_set: torch.Tensor,
    y: torch.Tensor,
    w: torch.Tensor,
    alpha_hat: float,
    beta_hat: np.ndarray,
    max_n: int,
    seed: int,
    ridge: float = 1e-8,
):
    N = emu_set.shape[0]
    rng = np.random.default_rng(seed)

    if N > max_n:
        idx = rng.choice(N, size=max_n, replace=False)
        idx = torch.tensor(idx, dtype=torch.long, device=DEVICE)
        emu = emu_set[idx]
        X   = X_set[idx]
        yy  = y[idx]
        ww  = w[idx]
    else:
        emu, X, yy, ww = emu_set, X_set, y, w

    sum_w = float(ww.sum().detach().cpu())
    Pdim = X.shape[2]

    alpha = torch.tensor([alpha_hat], dtype=DTYPE, device=DEVICE, requires_grad=True)
    beta  = torch.tensor(beta_hat, dtype=DTYPE, device=DEVICE, requires_grad=True)

    def loss_avg(a, b):
        V = a * emu + torch.einsum("bjp,p->bj", X, b)
        logP = V - torch.logsumexp(V, dim=1, keepdim=True)
        ll = logP[torch.arange(V.shape[0], device=DEVICE), yy]
        return (-(ww * ll)).sum() / (ww.sum() + 1e-12)

    L = loss_avg(alpha, beta)
    g = torch.autograd.grad(L, [alpha, beta], create_graph=True)
    g_vec = torch.cat([g[0].reshape(1), g[1].reshape(-1)], dim=0)

    H_avg = torch.zeros((1 + Pdim, 1 + Pdim), dtype=torch.float64, device=DEVICE)
    for i in range(1 + Pdim):
        gi = g_vec[i]
        hi = torch.autograd.grad(gi, [alpha, beta], retain_graph=True)
        hi_vec = torch.cat([hi[0].reshape(1), hi[1].reshape(-1)], dim=0)
        H_avg[i, :] = hi_vec.detach().to(torch.float64)

    H_avg = H_avg.cpu().numpy()
    H_sum = sum_w * H_avg

    H_sum = 0.5 * (H_sum + H_sum.T)
    H_sum = H_sum + ridge * np.eye(H_sum.shape[0], dtype=np.float64)

    Vcov = np.linalg.pinv(H_sum)
    se = np.sqrt(np.clip(np.diag(Vcov), 0.0, np.inf))
    return se


# ============================================================
# Plots
# ============================================================
def plot_distance_plausibility(obs_dist, exp_dist, model, out_png):
    obs_dist = np.asarray(obs_dist, dtype=np.float64)
    exp_dist = np.asarray(exp_dist, dtype=np.float64)

    finite = np.isfinite(obs_dist) & np.isfinite(exp_dist)
    obs_dist = obs_dist[finite]
    exp_dist = exp_dist[finite]

    if len(obs_dist) == 0:
        return

    obs_mean = float(np.mean(obs_dist))
    exp_mean = float(np.mean(exp_dist))

    xmax = float(max(np.max(obs_dist), np.max(exp_dist)))
    bins = np.linspace(0.0, xmax, 51)

    plt.figure()
    plt.hist(obs_dist, bins=bins, density=True, alpha=0.6,
             label=f"Observed dist_km (mean={obs_mean:.2f})")
    plt.hist(exp_dist, bins=bins, density=True, alpha=0.6,
             label=f"Predicted E[distance] (mean={exp_mean:.2f})")
    plt.xlabel("Distance (km)")
    plt.ylabel("Density")
    plt.title(f"Distance plausibility | model={model} | obs mean={obs_mean:.2f}, pred mean={exp_mean:.2f}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def plot_share_maps(tz_gdf, col, model, out_png, title):
    plt.figure()
    ax = tz_gdf.plot(column=col, legend=True)
    ax.set_axis_off()
    plt.title(f"{title} | model={model}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()


# ============================================================
# MAIN
# ============================================================
set_all_seeds(BASE_SEED)

print("[LOAD] TZ features (withLIE)...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)
if "npvm_id" not in tz.columns:
    raise ValueError("TZ layer missing 'npvm_id'")
tz["npvm_id"] = pd.to_numeric(tz["npvm_id"], errors="coerce")
tz = tz.dropna(subset=["npvm_id"]).copy()
tz["npvm_id"] = tz["npvm_id"].astype(int)

need_cols = ["npvm_id"] + BASE_COLS
miss = [c for c in need_cols if c not in tz.columns]
if miss:
    raise ValueError(f"TZ layer missing {miss}")

print("[PREP] Global z-standardize destination variables on FULL TZ universe...")
tz_feat = tz.copy()
tz_feat, Z_COLS, MU_ALL, SD_ALL = standardize_all_zones(tz_feat, BASE_COLS, suffix="_z")

print("[LOAD] EMU matrix + mapping...")
emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)
print(f"[OMX] mapping size={len(idx_to_zone)} | emu_mat shape={emu_mat.shape}")

print("[LOAD] Average distance matrix (for distance plausibility)...")
dist_mat, zone_to_idx_dist, idx_to_zone_dist = load_matrix_and_mapping(DIST_OMX, DIST_MAT_NAME, MAP_NAME)
if len(idx_to_zone_dist) != len(idx_to_zone) or not np.all(idx_to_zone_dist == idx_to_zone):
    print("[WARN] Distance OMX mapping differs from utilities OMX mapping. Distance plausibility may be misaligned.")

rows_sampled = []
rows_full    = []
rows_coef    = []

GPKG_OUT = os.path.join(OUT_DIR, "tz_outputs.gpkg")
if os.path.exists(GPKG_OUT):
    os.remove(GPKG_OUT)

for model in MODELS:
    print("\n" + "#" * 120)
    print(f"# MODEL = {model} | sources={MODEL_TO_SOURCE_SEGS[model]} | K={K} | epochs={EPOCHS}")
    print("#" * 120)

    train_df = build_pooled_dataset(TRIPS_DIR, model, "train")
    oos_df   = build_pooled_dataset(TRIPS_DIR, model, "oos")

    seed_tr   = BASE_SEED + 1000 * MODEL_SEED[model] + 10
    seed_te   = BASE_SEED + 1000 * MODEL_SEED[model] + 20
    seed_full = FULL_EVAL_SEED + 1000 * MODEL_SEED[model]

    emu_tr, X_tr, y_tr, w_tr, train_df2, _, _, _ = build_design(
        train_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_tr
    )
    emu_te, X_te, y_te, w_te, oos_df2, _, _, _ = build_design(
        oos_df, tz_feat, Z_COLS, emu_mat, zone_to_idx, k=K, seed=seed_te
    )

    print(f"[DATA] Train rows={len(train_df2):,} | OOS rows={len(oos_df2):,}")
    print(f"[DATA] Train WP={train_df2[WP_COL].sum():.2f} | OOS WP={oos_df2[WP_COL].sum():.2f}")

    alpha_hat, beta_hat = train_mnl(
        emu_tr, X_tr, y_tr, w_tr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY
    )

    met_tr = eval_sampled_metrics(emu_tr, X_tr, w_tr, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)
    met_te = eval_sampled_metrics(emu_te, X_te, w_te, alpha_hat, beta_hat, topKs=TOPKS_SAMPLED, want_ff=True)

    rows_sampled.append({"model": model, "split": "train", "alpha_emu": float(alpha_hat), "K": K, **met_tr})
    rows_sampled.append({"model": model, "split": "oos",   "alpha_emu": float(alpha_hat), "K": K, **met_te})

    print(
        "[SAMPLED OOS ]",
        f"NLL_sum={met_te['NLL_sum']:.2f} NLL_avg={met_te['NLL_avg']:.4f} "
        f"MRR={met_te['MRR']:.4f} Top5={met_te['Top5']:.3f} Top10={met_te['Top10']:.3f} FF={met_te['FF']:.4f}"
    )

    max_n = FULL_EVAL_MAX.get(model, None)
    full_out, share_pack, dist_pack, trip_distance_df = full_eval_and_shares_oos_subset(
        oos_df=oos_df2,
        tz_feat=tz_feat,
        z_cols=Z_COLS,
        emu_mat=emu_mat,
        dist_mat=dist_mat,
        zone_to_idx=zone_to_idx,
        idx_to_zone=idx_to_zone,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=max_n,
        seed=seed_full,
        chunk=FULL_EVAL_CHUNK,
        topKs=TOPKS_FULL,
    )

    rows_full.append({"model": model, "alpha_emu": float(alpha_hat), "K": K, **full_out})

    print(
        "[FULL OOS   ]",
        f"NLL_full_avg={full_out['NLL_full_avg']:.4f} MRR_full={full_out['MRR_full']:.4f} "
        f"FF_full={full_out['FF_full']:.6f} "
        f"R2_full={full_out['McFadden_R2_full']:.4f} Top5_full={full_out['Top5_full']:.3f} Top10_full={full_out['Top10_full']:.3f}"
    )
    print(
        "[SHARES FULL]",
        f"Pear={full_out['Pearson_share']:.4f} Spear={full_out['Spearman_share']:.4f} "
        f"MAE={full_out['MAE_share']:.6f} RMSE={full_out['RMSE_share']:.6f} JS={full_out['JS_share']:.6f}"
    )

    obs_dist, exp_dist = dist_pack
    if (obs_dist is not None) and (exp_dist is not None):
        png = os.path.join(PLOT_DIR, f"distance_plausibility_{model}.png")
        plot_distance_plausibility(obs_dist, exp_dist, model, png)
        print(f"[PLOT] wrote {png}")

        if trip_distance_df is not None:
            dist_csv = os.path.join(DIST_OOS_DIR, f"distance_triplevel_{model}.csv")
            trip_distance_df.to_csv(dist_csv, index=False)
            print(f"[CSV ] wrote {dist_csv}")

    se_vec = approx_hessian_se(
        emu_set=emu_tr,
        X_set=X_tr,
        y=y_tr,
        w=w_tr,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        max_n=SE_MAX_TRAIN.get(model, 20000),
        seed=SE_SEED + 1000 * MODEL_SEED[model],
        ridge=1e-8,
    )

    se_alpha = float(se_vec[0])
    se_beta  = se_vec[1:]

    zval = alpha_hat / (se_alpha + 1e-12)
    pval = p_value_two_sided_z(zval)
    rows_coef.append({
        "model": model,
        "param": "alpha_emu",
        "coef": float(alpha_hat),
        "se": float(se_alpha),
        "z": float(zval),
        "p": float(pval),
        "stars": stars(pval),
    })

    for name, b, s in zip(BASE_COLS, beta_hat.tolist(), se_beta.tolist()):
        zval = b / (s + 1e-12)
        pval = p_value_two_sided_z(zval)
        rows_coef.append({
            "model": model,
            "param": name,
            "coef": float(b),
            "se": float(s),
            "z": float(zval),
            "p": float(pval),
            "stars": stars(pval),
        })

    if share_pack is not None:
        tz_ids = share_pack["tz_ids"]
        S_obs  = share_pack["S_obs"]
        S_pred = share_pack["S_pred"]
        Aj     = share_pack["Aj"]

        tz_out = tz_feat[["npvm_id", "geometry"]].copy()
        tz_out = tz_out.set_index("npvm_id").loc[tz_ids].reset_index()

        tz_out[f"A_attr_{model}"] = Aj
        tz_out[f"S_obs_{model}"]  = S_obs
        tz_out[f"S_pred_{model}"] = S_pred

        png1 = os.path.join(PLOT_DIR, f"map_S_pred_{model}.png")
        png2 = os.path.join(PLOT_DIR, f"map_A_attr_{model}.png")
        plot_share_maps(tz_out, f"S_pred_{model}", model, png1, "Predicted destination share (OOS subset, full denom)")
        plot_share_maps(tz_out, f"A_attr_{model}", model, png2, "Attractivity index A_j = X_j beta (z-scale)")
        print(f"[PLOT] wrote {png1}")
        print(f"[PLOT] wrote {png2}")

        layer_name = f"TZ_{model}"
        tz_out.to_file(GPKG_OUT, layer=layer_name, driver="GPKG")
        print(f"[GPKG] wrote layer {layer_name} -> {GPKG_OUT}")


# ============================================================
# Save summary tables
# ============================================================
df_sampled = pd.DataFrame(rows_sampled)
df_full    = pd.DataFrame(rows_full)
df_coef    = pd.DataFrame(rows_coef)

p1 = os.path.join(OUT_DIR, "metrics_sampled.csv")
p2 = os.path.join(OUT_DIR, "metrics_full_and_shares.csv")
p3 = os.path.join(OUT_DIR, "coef_table.csv")

df_sampled.to_csv(p1, index=False)
df_full.to_csv(p2, index=False)
df_coef.to_csv(p3, index=False)

print("\n" + "=" * 110)
print("[OK] wrote", p1)
print("[OK] wrote", p2)
print("[OK] wrote", p3)
print("[OK] wrote", GPKG_OUT)
print("[OK] plots in", PLOT_DIR)
print("[OK] trip-level distance CSVs in", DIST_OOS_DIR)
print("=" * 110)

print("\nSAMPLED OOS METRICS (weighted):")
print(
    df_sampled[df_sampled["split"] == "oos"][
        ["model", "NLL_sum", "NLL_avg", "MRR", "Top5", "Top10", "FF", "N", "J", "sum_WP"]
    ]
    .sort_values("model")
    .to_string(index=False)
)

print("\nFULL OOS METRICS + SHARE DIAGNOSTICS:")
keep_cols = [
    "model", "NLL_full_sum", "NLL_full_avg", "MRR_full", "FF_full", "McFadden_R2_full", "Top5_full", "Top10_full",
    "Pearson_share", "Spearman_share", "MAE_share", "RMSE_share", "JS_share",
    "N_full_eval", "WP_full_eval", "J_full",
    "dist_obs_mean_km", "dist_pred_mean_km", "dist_mean_diff_km",
    "dist_obs_p50_km", "dist_pred_p50_km", "dist_obs_p90_km", "dist_pred_p90_km",
]
cols_exist = [c for c in keep_cols if c in df_full.columns]
print(df_full[cols_exist].sort_values("model").to_string(index=False))

## Odds-ratio exploration (proposed specification only)

In [ ]:
# ============================================================
# FIXED: interpret 1 z-score unit in ORIGINAL raw units
# when model uses z(log1p(raw)) (global).
#
# It will FAIL FAST if raw is not numeric, and prints diagnostics.
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

TZ_LOG1P_GPKG  = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LOG1P_LAYER = "TZ_first_sel_log1p"

TZ_RAW_GPKG  = "VariableAnalysis/TZ_sel.gpkg"
TZ_RAW_LAYER = "TZ_sel"

OUT_DIR = "ModelRuns/AgeBinsShortBaseline"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, "interp_1sd_rawunits_FIXED.csv")

BASE_COLS = [
    "F1_gastr_count_log1p",
    "F2_pop_total_log1p",
    "F4_cult_count_log1p",
    "F5_sport_count_log1p",
    "F5_sport_out_length_log1p",
    "F8_outdoor_lake_raw_dens_log1p",
    "F3_outdoor_hard_count_log1p",
    "F3_outdoor_soft_count_log1p",
    "F3_outdoor_LUmix",
    "F6_others_count_log1p",
    "F10_superinfra_log1p",
    "F7_urban_sum_log1p",
    "F8_POI_urban_dens_log1p",
]

def to_num_strict(s: pd.Series, name: str) -> pd.Series:
    """
    Convert to numeric robustly:
    - handles commas as decimal separators
    - strips spaces
    - reports how many become NaN
    """
    # already numeric?
    if pd.api.types.is_numeric_dtype(s):
        out = s.astype(float)
        return out

    # convert typical string issues
    ss = s.astype(str).str.strip()

    # if you have decimal comma, swap
    # (safe because counts usually have no commas, dens might)
    ss = ss.str.replace(",", ".", regex=False)

    out = pd.to_numeric(ss, errors="coerce")

    n = len(out)
    n_nan = int(out.isna().sum())
    n_fin = int(np.isfinite(out).sum())

    print(f"[RAW NUM] {name}: dtype_in={s.dtype} | finite={n_fin}/{n} | NaN={n_nan}/{n}")

    # if basically everything died, stop immediately
    if n_fin < max(50, 0.05 * n):
        # print a few examples to debug
        print(f"  [SAMPLE RAW STRINGS] {name}:",
              s.dropna().astype(str).head(10).tolist())
        raise ValueError(
            f"{name}: conversion to numeric failed (too many NaN). "
            "Your raw file likely stores values in a weird format."
        )

    return out.astype(float)

def expm1_safe(x):
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -700, 700)
    return np.expm1(x)

def q(x, p):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    return float(np.quantile(x, p))

print("[LOAD] log1p layer...")
tz_log = gpd.read_file(TZ_LOG1P_GPKG, layer=TZ_LOG1P_LAYER)
tz_log["npvm_id"] = pd.to_numeric(tz_log["npvm_id"], errors="coerce")
tz_log = tz_log.dropna(subset=["npvm_id"]).copy()
tz_log["npvm_id"] = tz_log["npvm_id"].astype(int)

print("[LOAD] raw layer...")
tz_raw = gpd.read_file(TZ_RAW_GPKG, layer=TZ_RAW_LAYER)
tz_raw["npvm_id"] = pd.to_numeric(tz_raw["npvm_id"], errors="coerce")
tz_raw = tz_raw.dropna(subset=["npvm_id"]).copy()
tz_raw["npvm_id"] = tz_raw["npvm_id"].astype(int)

# raw columns needed
raw_cols = sorted({c.replace("_log1p", "") for c in BASE_COLS if c.endswith("_log1p")})
keep_raw = ["npvm_id"] + [c for c in raw_cols if c in tz_raw.columns]
missing_raw = sorted(set(raw_cols) - set(keep_raw))
if missing_raw:
    raise ValueError(f"Missing raw columns in raw file: {missing_raw}")

# keep only needed from log1p
need_log = ["npvm_id"] + [c for c in BASE_COLS if c.endswith("_log1p")]
missing_log = [c for c in need_log if c not in tz_log.columns]
if missing_log:
    raise ValueError(f"Missing log1p columns in log file: {missing_log}")

tz = tz_log[need_log].merge(tz_raw[keep_raw], on="npvm_id", how="left", validate="1:1")
print(f"[OK] merged rows={len(tz):,}")

rows = []

for col_log in [c for c in BASE_COLS if c.endswith("_log1p")]:
    col_raw = col_log.replace("_log1p", "")

    # log1p series (should already be numeric, but be safe)
    xlog = pd.to_numeric(tz[col_log], errors="coerce").to_numpy(dtype=np.float64)
    xlog = xlog[np.isfinite(xlog)]
    if len(xlog) == 0:
        raise ValueError(f"{col_log}: no finite values")

    mu = float(np.mean(xlog))
    sd = float(np.std(xlog, ddof=0))
    if not np.isfinite(sd) or sd <= 0:
        raise ValueError(f"{col_log}: sd invalid ({sd})")

    # raw series (strict conversion + diagnostics)
    xraw = to_num_strict(tz[col_raw], col_raw).to_numpy(dtype=np.float64)
    xraw = xraw[np.isfinite(xraw)]
    xraw = np.clip(xraw, 0, np.inf)
    if len(xraw) == 0:
        raise ValueError(f"{col_raw}: no finite raw values after conversion")

    for qq in [0.10, 0.30, 0.50, 0.70, 0.90]:
        base_raw = q(xraw, qq)
        base_log = float(np.log1p(base_raw))

        # +1 z unit = +1 SD in log1p space
        new_log = base_log + sd
        new_raw = float(expm1_safe(new_log))
        delta_raw = new_raw - base_raw

        rows.append({
            "var": col_raw,
            "log_col": col_log,
            "sd_log1p": sd,
            "q": qq,
            "baseline_raw": base_raw,
            "raw_plus_1z": new_raw,
            "delta_raw_plus_1z": delta_raw,
            "interpretation": "Increase of +1 in z(log1p(raw))",
        })

# non-log1p feature
if "F3_outdoor_LUmix" in BASE_COLS:
    c = "F3_outdoor_LUmix"
    if c in tz_raw.columns:
        x = to_num_strict(tz_raw[c], c).to_numpy(dtype=np.float64)
        x = x[np.isfinite(x)]
        mu = float(np.mean(x))
        sd = float(np.std(x, ddof=0))
        for qq in [0.10, 0.50, 0.90]:
            base = q(x, qq)
            rows.append({
                "var": c,
                "log_col": c,
                "sd_log1p": sd,
                "q": qq,
                "baseline_raw": base,
                "raw_plus_1z": base + sd,
                "delta_raw_plus_1z": sd,
                "interpretation": "Not log1p: +1 SD on its own scale",
            })

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print("[OK] wrote", OUT_CSV)

print("\n[PREVIEW]")
print(df.sort_values(["var", "q"]).to_string(index=False))

import math

# Coefficienti hardcoded aggiornati
coefs = {
    "F5_sport_count": {
        "YS": 0.112,
        "OS": 0.053,
        "YL": -0.060,
        "OL": -0.155,
    },
    "F10_superinfra": {
        "YS": 0.181,
        "OS": 0.220,
        "YL": 0.265,
        "OL": 0.251,
    },
}

def odds_multiplier(beta: float) -> float:
    return math.exp(beta)

def odds_percent_change(beta: float) -> float:
    return (math.exp(beta) - 1.0) * 100.0

def compare_segments(var_name: str, seg1: str, seg2: str) -> None:
    b1 = coefs[var_name][seg1]
    b2 = coefs[var_name][seg2]

    om1 = odds_multiplier(b1)
    om2 = odds_multiplier(b2)

    ratio = om1 / om2
    pct_diff = (ratio - 1.0) * 100.0

    print(f"\n=== {var_name}: {seg1} vs {seg2} ===")
    print(f"{seg1}: beta = {b1:.3f} | exp(beta) = {om1:.3f} | odds change = {odds_percent_change(b1):.1f}%")
    print(f"{seg2}: beta = {b2:.3f} | exp(beta) = {om2:.3f} | odds change = {odds_percent_change(b2):.1f}%")
    print(f"Ratio of odds multipliers ({seg1}/{seg2}) = {ratio:.3f}")
    print(f"Interpretation: the odds effect is about {pct_diff:.1f}% stronger in {seg1} than in {seg2}.")

def print_all() -> None:
    print("Odds-ratio interpretation for +1 SD increase in the transformed covariate")
    print("-" * 75)

    for var_name, segs in coefs.items():
        print(f"\n### {var_name}")
        for seg, beta in segs.items():
            om = odds_multiplier(beta)
            pct = odds_percent_change(beta)
            print(f"{seg}: beta = {beta:.3f} | exp(beta) = {om:.3f} | odds change = {pct:.1f}%")

if __name__ == "__main__":
    print_all()

    # Confronti utili
    compare_segments("F5_sport_count", "YS", "OS")
    compare_segments("F5_sport_count", "YL", "OL")
    compare_segments("F5_sport_count", "YS", "YL")
    compare_segments("F10_superinfra", "YS", "OS")
    compare_segments("F10_superinfra", "YL", "YS")